# Stage 2: Constraint Extraction — Juris-Clarify Pipeline
## NER + Obligation Classification + CUAD Category Detection

**Architecture**: JointConstraintModel — Legal-BERT backbone with 3 heads:
1. **NER Head** (13 BIO labels + CRF): PARTY, DATE, MONEY, DURATION, JURISDICTION, LEGAL_REF
2. **Obligation Head** (5 types, BCE): MANDATORY, PROHIBITIVE, PERMISSIVE, CONDITIONAL, DECLARATIVE
3. **CUAD Category Head** (41-way, BCE): Full CUAD v1 clause taxonomy

**Data**: CUAD_v1.json gold character-level spans + Stage 1 clause segmentation outputs

---
### Kaggle Setup
1. Upload **CUAD_v1** as a Kaggle dataset (contains `CUAD_v1.json`)
2. Upload your **Stage 1 output folder** as a dataset (contains `train.jsonl`, `val.jsonl`, `test.jsonl`)
3. Enable **GPU** (T4 or P100) in Kaggle notebook settings


In [ ]:
!pip install -q pytorch-crf transformers

## Configuration
Adjust paths and hyperparameters below to match your Kaggle dataset names.

In [ ]:
import os

# ═══════════════════════════════════════════════════════════════════════════════
# PATHS — Update these to match your Kaggle dataset names
# ═══════════════════════════════════════════════════════════════════════════════
CUAD_JSON = "/kaggle/input/cuad-v1/CUAD_v1/CUAD_v1/CUAD_v1.json"

# Stage 1 PREDICTED segments (upload the predicted/ folder as a dataset named "stage1-predictions")
TRAIN_JSONL = "/kaggle/input/stage1-predictions/predicted_train.jsonl"
VAL_JSONL   = "/kaggle/input/stage1-predictions/predicted_val.jsonl"
TEST_JSONL  = "/kaggle/input/stage1-predictions/predicted_test.jsonl"
OUTPUT_DIR  = "/kaggle/working/stage2_output"

# ═══════════════════════════════════════════════════════════════════════════════
# HYPERPARAMETERS
# ═══════════════════════════════════════════════════════════════════════════════
ENCODER_MODEL    = "nlpaueb/legal-bert-base-uncased"
EPOCHS           = 4
BATCH_SIZE       = 8       # Kaggle T4 (15GB) can handle 8; use 4 for safety
MAX_SEQ_LENGTH   = 256
ENCODER_LR       = 2e-5
HEAD_LR          = 5e-4
WEIGHT_DECAY     = 0.01
FREEZE_EPOCHS    = 1
GRAD_ACCUM       = 2
VAL_SPLIT        = 0.15   # Only used if NOT using pre-split JSONL
PATIENCE         = 3
USE_FP16         = True    # Mixed precision — saves VRAM
SEED             = 42

os.makedirs(OUTPUT_DIR, exist_ok=True)
print("Configuration set.")
print(f"  CUAD JSON:    {CUAD_JSON}")
print(f"  Train JSONL:  {TRAIN_JSONL}")
print(f"  Val JSONL:    {VAL_JSONL}")
print(f"  Output dir:   {OUTPUT_DIR}")

## 1. Constraint Types & Taxonomy
Defines entity types, obligation types, CUAD categories, BIO label set, and structured output dataclasses.

In [ ]:
"""
constraint_types.py
-------------------
Defines the constraint taxonomy for Stage 2: Constraint Extraction.

Grounded in CUAD v1 (41 clause categories with character-level spans)
and LEDGAR (provision-level clause type labels).

Constraint Components:
  1. Named Entities   — PARTY, DATE, MONEY, DURATION, JURISDICTION, LEGAL_REF
  2. Obligation Types  — MANDATORY, PROHIBITIVE, PERMISSIVE, CONDITIONAL, DECLARATIVE
  3. CUAD Categories   — 41 clause types from the CUAD annotation scheme
"""

from dataclasses import dataclass, field
from enum import Enum
from typing import Dict, List, Optional, Tuple


# ── Named Entity Types ────────────────────────────────────────────────────────

class EntityType(str, Enum):
    PARTY        = "PARTY"         # company / person names — CUAD "Parties" spans
    DATE         = "DATE"          # agreement/effective/expiration dates — CUAD spans
    MONEY        = "MONEY"         # monetary values, caps, fees, percentages — regex
    DURATION     = "DURATION"      # "12 months", "thirty (30) days" — CUAD spans + regex
    JURISDICTION = "JURISDICTION"  # state/country for governing law — CUAD "Governing Law" spans
    LEGAL_REF    = "LEGAL_REF"     # Section 4.2, Exhibit A — regex


# BIO tag set for token-level NER
NER_LABELS = ["O"]
for etype in EntityType:
    NER_LABELS.append(f"B-{etype.value}")
    NER_LABELS.append(f"I-{etype.value}")

NER_LABEL2ID = {label: i for i, label in enumerate(NER_LABELS)}
NER_ID2LABEL = {i: label for label, i in NER_LABEL2ID.items()}
NUM_NER_LABELS = len(NER_LABELS)   # 13  (O + 6 entity types × 2 BIO)


# ── CUAD 41 Category Enumeration ─────────────────────────────────────────────
# These are the exact clause categories annotated in CUAD v1.
# Each has character-level answer spans in CUAD_v1.json (SQuAD format).

class CUADCategory(str, Enum):
    DOCUMENT_NAME                   = "Document Name"
    PARTIES                         = "Parties"
    AGREEMENT_DATE                  = "Agreement Date"
    EFFECTIVE_DATE                  = "Effective Date"
    EXPIRATION_DATE                 = "Expiration Date"
    RENEWAL_TERM                    = "Renewal Term"
    NOTICE_PERIOD_TO_TERMINATE      = "Notice Period To Terminate Renewal"
    GOVERNING_LAW                   = "Governing Law"
    MOST_FAVORED_NATION             = "Most Favored Nation"
    COMPETITIVE_RESTRICTION_EXCEPT  = "Competitive Restriction Exception"
    NON_COMPETE                     = "Non-Compete"
    EXCLUSIVITY                     = "Exclusivity"
    NO_SOLICIT_CUSTOMERS            = "No-Solicit Of Customers"
    NO_SOLICIT_EMPLOYEES            = "No-Solicit Of Employees"
    NON_DISPARAGEMENT               = "Non-Disparagement"
    TERMINATION_FOR_CONVENIENCE     = "Termination For Convenience"
    ROFR_ROFO_ROFN                  = "Rofr/Rofo/Rofn"
    CHANGE_OF_CONTROL               = "Change Of Control"
    ANTI_ASSIGNMENT                 = "Anti-Assignment"
    REVENUE_PROFIT_SHARING          = "Revenue/Profit Sharing"
    PRICE_RESTRICTIONS              = "Price Restrictions"
    MINIMUM_COMMITMENT              = "Minimum Commitment"
    VOLUME_RESTRICTION              = "Volume Restriction"
    IP_OWNERSHIP_ASSIGNMENT         = "Ip Ownership Assignment"
    JOINT_IP_OWNERSHIP              = "Joint Ip Ownership"
    LICENSE_GRANT                   = "License Grant"
    NON_TRANSFERABLE_LICENSE        = "Non-Transferable License"
    AFFILIATE_LICENSE_LICENSOR      = "Affiliate License-Licensor"
    AFFILIATE_LICENSE_LICENSEE      = "Affiliate License-Licensee"
    UNLIMITED_LICENSE               = "Unlimited/All-You-Can-Eat-License"
    IRREVOCABLE_PERPETUAL_LICENSE   = "Irrevocable Or Perpetual License"
    SOURCE_CODE_ESCROW              = "Source Code Escrow"
    POST_TERMINATION_SERVICES       = "Post-Termination Services"
    AUDIT_RIGHTS                    = "Audit Rights"
    UNCAPPED_LIABILITY              = "Uncapped Liability"
    CAP_ON_LIABILITY                = "Cap On Liability"
    LIQUIDATED_DAMAGES              = "Liquidated Damages"
    WARRANTY_DURATION               = "Warranty Duration"
    INSURANCE                       = "Insurance"
    COVENANT_NOT_TO_SUE             = "Covenant Not To Sue"
    THIRD_PARTY_BENEFICIARY         = "Third Party Beneficiary"


CUAD_CATEGORIES = [c.value for c in CUADCategory]
CUAD_CATEGORY2ID = {c: i for i, c in enumerate(CUAD_CATEGORIES)}
CUAD_ID2CATEGORY = {i: c for c, i in CUAD_CATEGORY2ID.items()}
NUM_CUAD_CATEGORIES = len(CUAD_CATEGORIES)   # 41

# CUAD category groups (categories that share annotation context)
CUAD_GROUPS = {
    1: ["Agreement Date", "Effective Date", "Expiration Date",
        "Renewal Term", "Notice Period To Terminate Renewal"],
    2: ["Non-Compete", "Exclusivity", "No-Solicit Of Customers",
        "Competitive Restriction Exception"],
    3: ["Change Of Control", "Anti-Assignment"],
    4: ["License Grant", "Non-Transferable License",
        "Affiliate License-Licensor", "Affiliate License-Licensee",
        "Irrevocable Or Perpetual License"],
    5: ["Uncapped Liability", "Cap On Liability"],
}

# 8 entity-answer categories (have specific values, not just Yes/No)
CUAD_ENTITY_CATEGORIES = {
    "Document Name", "Parties", "Agreement Date", "Effective Date",
    "Expiration Date", "Renewal Term", "Notice Period To Terminate Renewal",
    "Governing Law", "Warranty Duration",
}

# 33 binary categories (answer is Yes/No)
CUAD_BINARY_CATEGORIES = set(CUAD_CATEGORIES) - CUAD_ENTITY_CATEGORIES


# ── Obligation Types ──────────────────────────────────────────────────────────

class ObligationType(str, Enum):
    MANDATORY   = "MANDATORY"     # shall, must, is required to, will
    PROHIBITIVE = "PROHIBITIVE"   # shall not, must not, may not, prohibited
    PERMISSIVE  = "PERMISSIVE"    # may, can, is entitled to, has the right
    CONDITIONAL = "CONDITIONAL"   # if, provided that, subject to, unless
    DECLARATIVE = "DECLARATIVE"   # defines, means, refers to, is


OBLIGATION_LABELS = [o.value for o in ObligationType]
OBLIGATION_LABEL2ID = {label: i for i, label in enumerate(OBLIGATION_LABELS)}
OBLIGATION_ID2LABEL = {i: label for label, i in OBLIGATION_LABEL2ID.items()}
NUM_OBLIGATION_LABELS = len(OBLIGATION_LABELS)   # 5


# ── Deontic keyword patterns for rule-based obligation detection ──────────────

OBLIGATION_KEYWORDS: Dict[str, List[str]] = {
    "MANDATORY": [
        "shall", "must", "is required to", "are required to",
        "will", "agrees to", "is obligated to", "undertakes to",
        "covenants to", "warrants that", "represents that",
    ],
    "PROHIBITIVE": [
        "shall not", "must not", "may not", "will not",
        "is prohibited from", "is not permitted to",
        "is not allowed to", "cannot", "is restricted from",
    ],
    "PERMISSIVE": [
        "may", "can", "is entitled to", "has the right to",
        "is permitted to", "is allowed to", "at its option",
        "at the discretion of", "is authorized to",
    ],
    "CONDITIONAL": [
        "if", "provided that", "subject to", "unless",
        "in the event that", "on the condition that",
        "contingent upon", "notwithstanding", "except as",
        "to the extent that", "in case of",
    ],
    "DECLARATIVE": [
        "means", "shall mean", "is defined as", "refers to",
        "includes", "does not include", "as used herein",
    ],
}


# ── CUAD Category → Entity Type Mapping ──────────────────────────────────────
# Maps CUAD categories to the entity types their *gold spans* represent.
# Spans from CUAD_v1.json (SQuAD format) provide character-level positions.
# Categories NOT listed here have sentence-level spans (used for obligation
# classification) rather than extractable entity mentions.

CUAD_TO_ENTITY_TYPE: Dict[str, List[EntityType]] = {
    # GOLD-SPAN entity categories — spans from CUAD_v1.json are actual entities
    "Parties":                            [EntityType.PARTY],
    "Agreement Date":                     [EntityType.DATE],
    "Effective Date":                     [EntityType.DATE],
    "Expiration Date":                    [EntityType.DATE],
    "Renewal Term":                       [EntityType.DURATION],
    "Notice Period To Terminate Renewal": [EntityType.DURATION],
    "Warranty Duration":                  [EntityType.DURATION],
    "Governing Law":                      [EntityType.JURISDICTION],
    # Categories whose spans often CONTAIN entities (used for weak supervision)
    "Revenue/Profit Sharing":             [EntityType.MONEY],
    "Price Restrictions":                 [EntityType.MONEY],
    "Minimum Commitment":                 [EntityType.MONEY],
    "Volume Restriction":                 [EntityType.MONEY],
    "Cap On Liability":                   [EntityType.MONEY],
    "Uncapped Liability":                 [EntityType.MONEY],
    "Liquidated Damages":                 [EntityType.MONEY],
    "Insurance":                          [EntityType.MONEY],
}

# CUAD categories that indicate obligation-heavy clauses
CUAD_TO_OBLIGATION_TYPE: Dict[str, ObligationType] = {
    "Non-Compete":                        ObligationType.PROHIBITIVE,
    "Exclusivity":                        ObligationType.MANDATORY,
    "No-Solicit Of Customers":            ObligationType.PROHIBITIVE,
    "No-Solicit Of Employees":            ObligationType.PROHIBITIVE,
    "Non-Disparagement":                  ObligationType.PROHIBITIVE,
    "Competitive Restriction Exception":  ObligationType.CONDITIONAL,
    "Anti-Assignment":                    ObligationType.PROHIBITIVE,
    "Termination For Convenience":        ObligationType.PERMISSIVE,
    "Most Favored Nation":                ObligationType.MANDATORY,
    "Rofr/Rofo/Rofn":                    ObligationType.MANDATORY,
    "Change Of Control":                  ObligationType.CONDITIONAL,
    "License Grant":                      ObligationType.PERMISSIVE,
    "Non-Transferable License":           ObligationType.PROHIBITIVE,
    "Irrevocable Or Perpetual License":   ObligationType.PERMISSIVE,
    "Audit Rights":                       ObligationType.PERMISSIVE,
    "Covenant Not To Sue":                ObligationType.PROHIBITIVE,
    "Post-Termination Services":          ObligationType.MANDATORY,
    "Third Party Beneficiary":            ObligationType.DECLARATIVE,
    "Ip Ownership Assignment":            ObligationType.MANDATORY,
    "Joint Ip Ownership":                 ObligationType.DECLARATIVE,
    "Source Code Escrow":                 ObligationType.CONDITIONAL,
}


# ── LEDGAR Label → CUAD Category Mapping ─────────────────────────────────────
# Maps top LEDGAR provision labels to the closest CUAD category.
# LEDGAR has 100K+ labels; we map only the high-frequency ones (~80) that
# have a clear CUAD counterpart.  Used for pre-training / data augmentation.

LEDGAR_TO_CUAD: Dict[str, str] = {
    "governing law":               "Governing Law",
    "applicable law":              "Governing Law",
    "choice of law":               "Governing Law",
    "termination":                 "Termination For Convenience",
    "term":                        "Renewal Term",
    "assignment":                  "Anti-Assignment",
    "successors and assigns":      "Anti-Assignment",
    "non-compete":                 "Non-Compete",
    "non-competition":             "Non-Compete",
    "exclusivity":                 "Exclusivity",
    "non-solicitation":            "No-Solicit Of Employees",
    "non-disparagement":           "Non-Disparagement",
    "change in control":           "Change Of Control",
    "change of control":           "Change Of Control",
    "insurance":                   "Insurance",
    "indemnification":             "Cap On Liability",
    "limitation of liability":     "Cap On Liability",
    "audit rights":                "Audit Rights",
    "audit":                       "Audit Rights",
    "intellectual property":       "Ip Ownership Assignment",
    "license grant":               "License Grant",
    "license":                     "License Grant",
    "confidentiality":             "Covenant Not To Sue",
    "revenue sharing":             "Revenue/Profit Sharing",
    "minimum commitment":          "Minimum Commitment",
    "price":                       "Price Restrictions",
    "pricing":                     "Price Restrictions",
    "warranty":                    "Warranty Duration",
    "liquidated damages":          "Liquidated Damages",
    "escrow":                      "Source Code Escrow",
    "third party beneficiary":     "Third Party Beneficiary",
    "third party beneficiaries":   "Third Party Beneficiary",
    # LEDGAR labels that don't map to CUAD but are useful for immutability
    "definitions":                 "_IMMUTABLE",
    "defined terms":               "_IMMUTABLE",
    "entire agreement":            "_IMMUTABLE",
    "severability":                "_IMMUTABLE",
    "counterparts":                "_IMMUTABLE",
    "headings":                    "_IMMUTABLE",
    "construction":                "_IMMUTABLE",
    "integration":                 "_IMMUTABLE",
    "interpretation":              "_IMMUTABLE",
}

# Reverse lookup: CUAD category → list of LEDGAR labels
CUAD_TO_LEDGAR: Dict[str, List[str]] = {}
for ledgar_label, cuad_cat in LEDGAR_TO_CUAD.items():
    CUAD_TO_LEDGAR.setdefault(cuad_cat, []).append(ledgar_label)


# ── Structured constraint output ──────────────────────────────────────────────

@dataclass
class ExtractedEntity:
    """A single named entity extracted from a clause."""
    entity_type: EntityType
    text: str
    start_char: int         # offset within the clause text
    end_char: int
    confidence: float = 1.0


@dataclass
class ObligationLabel:
    """Obligation classification for a sentence within a clause."""
    obligation_type: ObligationType
    confidence: float = 1.0
    trigger_phrase: str = ""   # the keyword(s) that caused this classification


@dataclass
class ClauseConstraints:
    """
    Full constraint profile for one clause segment.
    This is the output of Stage 2 and the input to Stage 3 (simplification).
    The simplifier must preserve ALL entities and respect obligation types.
    """
    clause_index: int
    clause_text: str
    start_char: int               # offset in original document
    end_char: int

    # Extracted entities (must be preserved verbatim in simplification)
    entities: List[ExtractedEntity] = field(default_factory=list)

    # Sentence-level obligation types
    obligations: List[ObligationLabel] = field(default_factory=list)

    # CUAD categories that apply to this clause (from Stage 1 multitask)
    cuad_categories: List[str] = field(default_factory=list)

    # Immutability flag — if True, this clause should NOT be simplified
    # (e.g., pure definitions, governing law references)
    immutable: bool = False

    @property
    def entity_texts(self) -> Dict[str, List[str]]:
        """Group extracted entities by type for quick lookup."""
        result: Dict[str, List[str]] = {}
        for e in self.entities:
            result.setdefault(e.entity_type.value, []).append(e.text)
        return result

    @property
    def has_monetary_constraints(self) -> bool:
        return any(e.entity_type == EntityType.MONEY for e in self.entities)

    @property
    def has_temporal_constraints(self) -> bool:
        return any(
            e.entity_type in (EntityType.DATE, EntityType.DURATION)
            for e in self.entities
        )

    @property
    def primary_obligation(self) -> Optional[ObligationType]:
        """Most frequent obligation type in this clause."""
        if not self.obligations:
            return None
        from collections import Counter
        counts = Counter(o.obligation_type for o in self.obligations)
        return counts.most_common(1)[0][0]

    def to_dict(self) -> dict:
        return {
            "clause_index": self.clause_index,
            "clause_text": self.clause_text,
            "start_char": self.start_char,
            "end_char": self.end_char,
            "entities": [
                {
                    "type": e.entity_type.value,
                    "text": e.text,
                    "start": e.start_char,
                    "end": e.end_char,
                    "confidence": round(e.confidence, 4),
                }
                for e in self.entities
            ],
            "obligations": [
                {
                    "type": o.obligation_type.value,
                    "confidence": round(o.confidence, 4),
                    "trigger": o.trigger_phrase,
                }
                for o in self.obligations
            ],
            "cuad_categories": self.cuad_categories,
            "immutable": self.immutable,
            "primary_obligation": (
                self.primary_obligation.value if self.primary_obligation else None
            ),
        }


@dataclass
class DocumentConstraints:
    """Constraint profiles for all clauses in a document."""
    doc_id: str
    clauses: List[ClauseConstraints]

    @property
    def all_parties(self) -> List[str]:
        parties = set()
        for c in self.clauses:
            for e in c.entities:
                if e.entity_type == EntityType.PARTY:
                    parties.add(e.text)
        return sorted(parties)

    @property
    def all_dates(self) -> List[str]:
        dates = set()
        for c in self.clauses:
            for e in c.entities:
                if e.entity_type == EntityType.DATE:
                    dates.add(e.text)
        return sorted(dates)

    @property
    def all_jurisdictions(self) -> List[str]:
        jurisdictions = set()
        for c in self.clauses:
            for e in c.entities:
                if e.entity_type == EntityType.JURISDICTION:
                    jurisdictions.add(e.text)
        return sorted(jurisdictions)

    @property
    def cuad_category_summary(self) -> Dict[str, int]:
        """Count of clauses per CUAD category across the document."""
        from collections import Counter
        counts: Counter = Counter()
        for c in self.clauses:
            for cat in c.cuad_categories:
                counts[cat] += 1
        return dict(counts.most_common())

    def to_dict(self) -> dict:
        return {
            "doc_id": self.doc_id,
            "num_clauses": len(self.clauses),
            "all_parties": self.all_parties,
            "all_dates": self.all_dates,
            "all_jurisdictions": self.all_jurisdictions,
            "cuad_category_summary": self.cuad_category_summary,
            "clauses": [c.to_dict() for c in self.clauses],
        }


## 2. Neural Models
- `ConstraintNERModel` — Token-level NER (Legal-BERT + linear + CRF)
- `ObligationClassifier` — Sentence-level obligation (CLS + MLP + BCE)
- `CUADCategoryClassifier` — Clause-level 41-way (CLS + MLP + BCE)
- `JointConstraintModel` — Shared backbone with all 3 heads

In [ ]:
"""
constraint_model.py
-------------------
Stage 2 models for constraint extraction:

  1. ConstraintNERModel   — Token-level NER using Legal-BERT to extract
                            PARTY, DATE, MONEY, DURATION, JURISDICTION, LEGAL_REF.
  2. ObligationClassifier — Sentence-level multi-label classifier for
                            MANDATORY / PROHIBITIVE / PERMISSIVE / CONDITIONAL /
                            DECLARATIVE obligation types.
  3. CUADCategoryClassifier — Clause-level 41-way multi-label classifier for
                            CUAD categories (grounded in gold annotations).
  4. JointConstraintModel — Combined model sharing a Legal-BERT backbone with
                            three heads (NER + obligation + CUAD category).

All models use the same Legal-BERT encoder as Stage 1 for consistency.
NER labels: 13 (O + 6 entity types × 2 BIO tags).
CUAD categories: 41 clause types from CUAD v1 gold annotations.
"""

from dataclasses import dataclass
from typing import Dict, List, Optional, Tuple

import torch
import torch.nn as nn
from transformers import AutoModel, PreTrainedModel

# constraint_types already defined above


# ── Config ────────────────────────────────────────────────────────────────────

@dataclass
class ConstraintModelConfig:
    # Backbone
    encoder_model_name: str = "nlpaueb/legal-bert-base-uncased"
    freeze_encoder_epochs: int = 1

    # NER head
    ner_num_labels: int = NUM_NER_LABELS       # 13 (O + 6×2 BIO)
    ner_dropout: float = 0.2
    use_ner_crf: bool = True

    # Obligation head
    obligation_num_labels: int = NUM_OBLIGATION_LABELS  # 5
    obligation_dropout: float = 0.2

    # CUAD category head
    category_num_labels: int = NUM_CUAD_CATEGORIES  # 41
    category_dropout: float = 0.2

    # Joint training loss weights
    obligation_loss_weight: float = 0.5   # λ_obl
    category_loss_weight: float = 0.3     # λ_cat
    max_seq_length: int = 256             # token-level max for NER


# ── NER Model (token classification) ─────────────────────────────────────────

class ConstraintNERModel(nn.Module):
    """
    Token-level NER for legal constraint entities.

    Input:  tokenised clause text → [batch, seq_len]
    Output: per-token BIO labels  → [batch, seq_len, NUM_NER_LABELS]
    """

    def __init__(self, cfg: ConstraintModelConfig):
        super().__init__()
        self.cfg = cfg
        self.encoder: PreTrainedModel = AutoModel.from_pretrained(
            cfg.encoder_model_name
        )
        hidden_size = self.encoder.config.hidden_size

        self.dropout = nn.Dropout(cfg.ner_dropout)
        self.classifier = nn.Linear(hidden_size, cfg.ner_num_labels)

        self.crf = None
        if cfg.use_ner_crf:
            from torchcrf import CRF
            self.crf = CRF(cfg.ner_num_labels, batch_first=True)

        self.loss_fn = nn.CrossEntropyLoss(ignore_index=-100)

    def forward(
        self,
        input_ids: torch.Tensor,         # [B, L]
        attention_mask: torch.Tensor,     # [B, L]
        labels: Optional[torch.Tensor] = None,  # [B, L]
    ) -> Tuple[Optional[torch.Tensor], torch.Tensor]:
        """Returns (loss, logits)."""
        outputs = self.encoder(
            input_ids=input_ids,
            attention_mask=attention_mask,
        )
        sequence_output = self.dropout(outputs.last_hidden_state)  # [B, L, H]
        logits = self.classifier(sequence_output)                  # [B, L, C]

        loss = None
        if labels is not None:
            if self.crf is not None:
                crf_mask = attention_mask.bool()
                safe_labels = labels.clone()
                safe_labels[safe_labels == -100] = 0
                loss = -self.crf(logits, safe_labels, mask=crf_mask, reduction="mean")
            else:
                B, L, C = logits.shape
                loss = self.loss_fn(logits.view(B * L, C), labels.view(B * L))

        return loss, logits

    @torch.no_grad()
    def predict(
        self,
        input_ids: torch.Tensor,
        attention_mask: torch.Tensor,
    ) -> torch.Tensor:
        """Returns predicted label IDs [B, L]."""
        _, logits = self.forward(input_ids, attention_mask)

        if self.crf is not None:
            preds = self.crf.decode(logits, mask=attention_mask.bool())
            B, L = logits.shape[:2]
            out = torch.zeros(B, L, dtype=torch.long, device=logits.device)
            for i, seq in enumerate(preds):
                out[i, : len(seq)] = torch.tensor(seq, device=logits.device)
            return out
        else:
            return logits.argmax(dim=-1)

    def freeze_encoder(self):
        for p in self.encoder.parameters():
            p.requires_grad = False

    def unfreeze_encoder(self):
        for p in self.encoder.parameters():
            p.requires_grad = True


# ── Obligation Classifier (sentence-level) ────────────────────────────────────

class ObligationClassifier(nn.Module):
    """
    Sentence-level obligation type classifier.

    Input:  tokenised sentence → [batch, seq_len]
    Output: obligation type    → [batch, NUM_OBLIGATION_LABELS]

    Multi-label: a sentence can be both CONDITIONAL and MANDATORY
    (e.g. "If X occurs, the Company shall pay Y").
    """

    def __init__(self, cfg: ConstraintModelConfig):
        super().__init__()
        self.cfg = cfg
        self.encoder: PreTrainedModel = AutoModel.from_pretrained(
            cfg.encoder_model_name
        )
        hidden_size = self.encoder.config.hidden_size

        self.head = nn.Sequential(
            nn.Dropout(cfg.obligation_dropout),
            nn.Linear(hidden_size, hidden_size // 2),
            nn.GELU(),
            nn.Dropout(cfg.obligation_dropout / 2),
            nn.Linear(hidden_size // 2, cfg.obligation_num_labels),
        )

        # Multi-label BCE loss
        self.loss_fn = nn.BCEWithLogitsLoss()

    def forward(
        self,
        input_ids: torch.Tensor,         # [B, L]
        attention_mask: torch.Tensor,     # [B, L]
        labels: Optional[torch.Tensor] = None,  # [B, num_obligation_labels]
    ) -> Tuple[Optional[torch.Tensor], torch.Tensor]:
        """Returns (loss, logits)."""
        outputs = self.encoder(
            input_ids=input_ids,
            attention_mask=attention_mask,
        )
        # Use CLS token as sentence representation
        cls_emb = outputs.last_hidden_state[:, 0, :]  # [B, H]
        logits = self.head(cls_emb)                    # [B, C]

        loss = None
        if labels is not None:
            loss = self.loss_fn(logits, labels.float())

        return loss, logits

    @torch.no_grad()
    def predict(
        self,
        input_ids: torch.Tensor,
        attention_mask: torch.Tensor,
        threshold: float = 0.5,
    ) -> torch.Tensor:
        """Returns multi-label predictions [B, C] as binary tensor."""
        _, logits = self.forward(input_ids, attention_mask)
        return (torch.sigmoid(logits) >= threshold).long()

    def freeze_encoder(self):
        for p in self.encoder.parameters():
            p.requires_grad = False

    def unfreeze_encoder(self):
        for p in self.encoder.parameters():
            p.requires_grad = True


# ── CUAD Category Classifier (clause-level, 41-way multi-label) ──────────────

class CUADCategoryClassifier(nn.Module):
    """
    Clause-level 41-way multi-label classifier for CUAD categories.
    Trained on gold annotations from CUAD_v1.json.
    Can also be pre-trained on LEDGAR provisions via category mapping.
    """

    def __init__(self, cfg: ConstraintModelConfig):
        super().__init__()
        self.cfg = cfg
        self.encoder: PreTrainedModel = AutoModel.from_pretrained(
            cfg.encoder_model_name
        )
        hidden_size = self.encoder.config.hidden_size

        self.head = nn.Sequential(
            nn.Dropout(cfg.category_dropout),
            nn.Linear(hidden_size, hidden_size // 2),
            nn.GELU(),
            nn.Dropout(cfg.category_dropout / 2),
            nn.Linear(hidden_size // 2, cfg.category_num_labels),
        )

        self.loss_fn = nn.BCEWithLogitsLoss()

    def forward(
        self,
        input_ids: torch.Tensor,
        attention_mask: torch.Tensor,
        labels: Optional[torch.Tensor] = None,
    ) -> Tuple[Optional[torch.Tensor], torch.Tensor]:
        outputs = self.encoder(input_ids=input_ids, attention_mask=attention_mask)
        cls_emb = outputs.last_hidden_state[:, 0, :]
        logits = self.head(cls_emb)

        loss = None
        if labels is not None:
            loss = self.loss_fn(logits, labels.float())

        return loss, logits

    @torch.no_grad()
    def predict(
        self,
        input_ids: torch.Tensor,
        attention_mask: torch.Tensor,
        threshold: float = 0.5,
    ) -> torch.Tensor:
        _, logits = self.forward(input_ids, attention_mask)
        return (torch.sigmoid(logits) >= threshold).long()

    def freeze_encoder(self):
        for p in self.encoder.parameters():
            p.requires_grad = False

    def unfreeze_encoder(self):
        for p in self.encoder.parameters():
            p.requires_grad = True


# ── Joint Constraint Model (shared backbone) ─────────────────────────────────

class JointConstraintModel(nn.Module):
    """
    Shared Legal-BERT backbone with three heads:
      1. Token-level NER head     → extracts PARTY, DATE, MONEY, DURATION,
                                     JURISDICTION, LEGAL_REF (13 BIO labels)
      2. Clause-level obligation  → classifies obligation type (5 labels)
      3. Clause-level CUAD category → classifies CUAD categories (41 labels)

    During training, all heads are optimised jointly:
      L = L_ner + λ_obl * L_obligation + λ_cat * L_category

    During inference, the model produces NER tags, obligation labels,
    and category predictions in a single forward pass.
    """

    def __init__(self, cfg: ConstraintModelConfig):
        super().__init__()
        self.cfg = cfg

        # Shared backbone
        self.encoder: PreTrainedModel = AutoModel.from_pretrained(
            cfg.encoder_model_name
        )
        hidden_size = self.encoder.config.hidden_size

        # ── NER head (token-level) ────────────────────────────────────────────
        self.ner_dropout = nn.Dropout(cfg.ner_dropout)
        self.ner_classifier = nn.Linear(hidden_size, cfg.ner_num_labels)

        self.ner_crf = None
        if cfg.use_ner_crf:
            from torchcrf import CRF
            self.ner_crf = CRF(cfg.ner_num_labels, batch_first=True)

        self.ner_loss_fn = nn.CrossEntropyLoss(ignore_index=-100)

        # ── Obligation head (clause-level) ────────────────────────────────────
        self.obligation_head = nn.Sequential(
            nn.Dropout(cfg.obligation_dropout),
            nn.Linear(hidden_size, hidden_size // 2),
            nn.GELU(),
            nn.Dropout(cfg.obligation_dropout / 2),
            nn.Linear(hidden_size // 2, cfg.obligation_num_labels),
        )
        self.obligation_loss_fn = nn.BCEWithLogitsLoss()

        # ── CUAD category head (clause-level, 41-way) ────────────────────────
        self.category_head = nn.Sequential(
            nn.Dropout(cfg.category_dropout),
            nn.Linear(hidden_size, hidden_size // 2),
            nn.GELU(),
            nn.Dropout(cfg.category_dropout / 2),
            nn.Linear(hidden_size // 2, cfg.category_num_labels),
        )
        self.category_loss_fn = nn.BCEWithLogitsLoss()

    def forward(
        self,
        input_ids: torch.Tensor,             # [B, L]
        attention_mask: torch.Tensor,         # [B, L]
        ner_labels: Optional[torch.Tensor] = None,         # [B, L]
        obligation_labels: Optional[torch.Tensor] = None,  # [B, 5]
        category_labels: Optional[torch.Tensor] = None,    # [B, 41]
    ) -> Dict[str, Optional[torch.Tensor]]:
        """
        Returns dict with keys:
          'loss'              : combined loss (if labels provided)
          'ner_logits'        : [B, L, ner_num_labels]
          'obligation_logits' : [B, obligation_num_labels]
          'category_logits'   : [B, category_num_labels]
        """
        outputs = self.encoder(
            input_ids=input_ids,
            attention_mask=attention_mask,
        )
        hidden_states = outputs.last_hidden_state  # [B, L, H]
        cls_emb = hidden_states[:, 0, :]           # [B, H]

        # ── NER ───────────────────────────────────────────────────────────────
        ner_out = self.ner_dropout(hidden_states)
        ner_logits = self.ner_classifier(ner_out)   # [B, L, C_ner]

        # ── Obligation ────────────────────────────────────────────────────────
        obligation_logits = self.obligation_head(cls_emb)  # [B, C_obl]

        # ── CUAD Category ─────────────────────────────────────────────────────
        category_logits = self.category_head(cls_emb)      # [B, C_cat]

        # ── Losses ────────────────────────────────────────────────────────────
        ner_loss = None
        if ner_labels is not None:
            if self.ner_crf is not None:
                crf_mask = attention_mask.bool()
                safe_labels = ner_labels.clone()
                safe_labels[safe_labels == -100] = 0
                ner_loss = -self.ner_crf(
                    ner_logits, safe_labels, mask=crf_mask, reduction="mean"
                )
            else:
                B, L, C = ner_logits.shape
                ner_loss = self.ner_loss_fn(
                    ner_logits.view(B * L, C), ner_labels.view(B * L)
                )

        obligation_loss = None
        if obligation_labels is not None:
            obligation_loss = self.obligation_loss_fn(
                obligation_logits, obligation_labels.float()
            )

        category_loss = None
        if category_labels is not None:
            category_loss = self.category_loss_fn(
                category_logits, category_labels.float()
            )

        combined_loss = None
        losses = []
        if ner_loss is not None:
            losses.append(ner_loss)
        if obligation_loss is not None:
            losses.append(self.cfg.obligation_loss_weight * obligation_loss)
        if category_loss is not None:
            losses.append(self.cfg.category_loss_weight * category_loss)
        if losses:
            combined_loss = sum(losses)

        return {
            "loss": combined_loss,
            "ner_logits": ner_logits,
            "obligation_logits": obligation_logits,
            "category_logits": category_logits,
        }

    @torch.no_grad()
    def predict(
        self,
        input_ids: torch.Tensor,
        attention_mask: torch.Tensor,
        obligation_threshold: float = 0.5,
        category_threshold: float = 0.5,
    ) -> Tuple[torch.Tensor, torch.Tensor, torch.Tensor]:
        """
        Returns:
          ner_preds        : [B, L]       token-level entity labels
          obligation_preds : [B, C_obl]   multi-label binary obligations
          category_preds   : [B, C_cat]   multi-label binary categories
        """
        result = self.forward(input_ids, attention_mask)

        # NER predictions
        ner_logits = result["ner_logits"]
        if self.ner_crf is not None:
            ner_decoded = self.ner_crf.decode(ner_logits, mask=attention_mask.bool())
            B, L = ner_logits.shape[:2]
            ner_preds = torch.zeros(B, L, dtype=torch.long, device=ner_logits.device)
            for i, seq in enumerate(ner_decoded):
                ner_preds[i, : len(seq)] = torch.tensor(seq, device=ner_logits.device)
        else:
            ner_preds = ner_logits.argmax(dim=-1)

        # Obligation predictions
        obligation_logits = result["obligation_logits"]
        obligation_preds = (
            torch.sigmoid(obligation_logits) >= obligation_threshold
        ).long()

        # Category predictions
        category_logits = result["category_logits"]
        category_preds = (
            torch.sigmoid(category_logits) >= category_threshold
        ).long()

        return ner_preds, obligation_preds, category_preds

    def freeze_encoder(self):
        for p in self.encoder.parameters():
            p.requires_grad = False

    def unfreeze_encoder(self):
        for p in self.encoder.parameters():
            p.requires_grad = True


## 3. Datasets & Data Loading
- CUAD gold span loader (character-level NER annotations)
- Regex patterns for silver-label NER (MONEY, LEGAL_REF)
- BIO label alignment to tokens
- NER / Obligation / Category / Joint dataset classes
- Collate functions

In [ ]:
"""
constraint_dataset.py
---------------------
Dataset and data-loading utilities for Stage 2: Constraint Extraction.

Data sources:
  1. CUAD_v1.json (SQuAD format) — provides gold character-level spans for
     entity extraction: Parties, Dates, Durations, Governing Law, etc.
  2. CUAD master_clauses.csv — provides per-contract Yes/No category labels
     → used for clause-level CUAD category classification
  3. Rule-based pattern matching — generates silver labels for MONEY and
     LEGAL_REF entities (not directly annotated in CUAD).
  4. LEDGAR (sec_corpus_2016-2019.jsonl) — provision-level clause type labels
     → used for pre-training / augmenting the clause classifier.

The dataset produces three types of training samples:
  (a) Token-level NER samples  — one per clause, with BIO tags (gold + silver)
  (b) Sentence-level obligation samples — one per sentence, multi-label
  (c) CUAD category classification samples — one per clause, 41-way multi-label
"""

import csv
import json
import logging
import re
from dataclasses import dataclass
from pathlib import Path
from typing import Dict, List, Optional, Tuple

import torch
from torch.utils.data import Dataset
from transformers import PreTrainedTokenizerBase

# constraint_types already defined above

log = logging.getLogger(__name__)


# ── Segment file loader (JSON array or JSONL) ────────────────────────────────

def load_segments(path: str | Path) -> List[Dict]:
    """
    Load segmented contracts from either a JSON array file or a JSONL file.
    Detects format automatically: if json.load succeeds and yields a list,
    treat as JSON array; otherwise read line-by-line as JSONL.
    """
    path = Path(path)
    with open(path, "r", encoding="utf-8") as f:
        first = f.read(1)
    if first == "[":
        with open(path, "r", encoding="utf-8") as f:
            data = json.load(f)
        if isinstance(data, list):
            return data
    # JSONL fallback
    records = []
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if line:
                records.append(json.loads(line))
    log.info(f"Loaded {len(records)} segment records from {path.name} (JSONL)")
    return records


# ── Regex patterns for silver-standard NER ────────────────────────────────────

# Date patterns: "January 1, 2024", "01/15/2024", "1st day of March, 2024"
DATE_PATTERNS = [
    re.compile(
        r"\b(?:January|February|March|April|May|June|July|August|September|"
        r"October|November|December)\s+\d{1,2},?\s+\d{4}\b",
        re.IGNORECASE,
    ),
    re.compile(r"\b\d{1,2}[/-]\d{1,2}[/-]\d{2,4}\b"),
    re.compile(
        r"\b\d{1,2}(?:st|nd|rd|th)\s+day\s+of\s+\w+,?\s+\d{4}\b",
        re.IGNORECASE,
    ),
    re.compile(r"\b\d{4}[-/]\d{2}[-/]\d{2}\b"),  # ISO format
]

# Money patterns: "$1,000,000", "USD 50,000", "10%", "fifty thousand dollars"
MONEY_PATTERNS = [
    re.compile(r"\$[\d,]+(?:\.\d{1,2})?(?:\s*(?:million|billion|thousand))?", re.IGNORECASE),
    re.compile(r"\b(?:USD|EUR|GBP|CAD)\s*[\d,]+(?:\.\d{1,2})?", re.IGNORECASE),
    re.compile(r"\b\d+(?:\.\d+)?%\b"),
    re.compile(
        r"\b(?:one|two|three|four|five|six|seven|eight|nine|ten|twenty|thirty|"
        r"forty|fifty|hundred|thousand|million|billion)\s+"
        r"(?:hundred|thousand|million|billion)?\s*"
        r"(?:dollars|pounds|euros)\b",
        re.IGNORECASE,
    ),
]

# Duration patterns: "twelve (12) months", "30 days", "five (5) years"
DURATION_PATTERNS = [
    re.compile(
        r"\b\w+\s*\(\d+\)\s*(?:days?|months?|years?|weeks?|business\s+days?)\b",
        re.IGNORECASE,
    ),
    re.compile(
        r"\b\d+\s*(?:days?|months?|years?|weeks?|calendar\s+days?|business\s+days?)\b",
        re.IGNORECASE,
    ),
    re.compile(
        r"\b(?:one|two|three|four|five|six|seven|eight|nine|ten|twelve|"
        r"twenty|thirty|sixty|ninety)\s+"
        r"(?:days?|months?|years?|weeks?)\b",
        re.IGNORECASE,
    ),
]

# Legal reference patterns: "Section 4.2", "Article III", "Exhibit A"
LEGAL_REF_PATTERNS = [
    re.compile(r"\bSection\s+\d+(?:\.\d+)*(?:\([a-z]\))?", re.IGNORECASE),
    re.compile(r"\bArticle\s+(?:\d+|[IVXLC]+)\b", re.IGNORECASE),
    re.compile(r"\bExhibit\s+[A-Z]\b", re.IGNORECASE),
    re.compile(r"\bSchedule\s+[A-Z\d]+\b", re.IGNORECASE),
    re.compile(r"\bAnnex\s+[A-Z\d]+\b", re.IGNORECASE),
    re.compile(r"\bAppendix\s+[A-Z\d]+\b", re.IGNORECASE),
]

ENTITY_REGEX_MAP: Dict[EntityType, List[re.Pattern]] = {
    EntityType.DATE: DATE_PATTERNS,
    EntityType.MONEY: MONEY_PATTERNS,
    EntityType.DURATION: DURATION_PATTERNS,
    EntityType.LEGAL_REF: LEGAL_REF_PATTERNS,
}


# ── CUAD v1 Gold Span Loader ─────────────────────────────────────────────────

# Mapping from CUAD question category name to the entity type its spans
# directly represent.  Only entity-extraction categories are listed —
# binary Yes/No categories provide sentence spans (used differently).
_CUAD_SPAN_ENTITY_MAP: Dict[str, EntityType] = {
    "Parties":                            EntityType.PARTY,
    "Agreement Date":                     EntityType.DATE,
    "Effective Date":                     EntityType.DATE,
    "Expiration Date":                    EntityType.DATE,
    "Renewal Term":                       EntityType.DURATION,
    "Notice Period To Terminate Renewal": EntityType.DURATION,
    "Warranty Duration":                  EntityType.DURATION,
    "Governing Law":                      EntityType.JURISDICTION,
}


def _extract_category_from_question(question: str) -> str:
    """Extract the quoted category name from a CUAD SQuAD-format question."""
    start = question.find('"')
    end = question.find('"', start + 1) if start >= 0 else -1
    if start >= 0 and end > start:
        return question[start + 1 : end]
    return ""


def load_cuad_gold_annotations(
    cuad_json_path: str | Path,
) -> Dict[str, Dict[str, object]]:
    """
    Load CUAD_v1.json and extract:
      - Per-document entity spans (character offsets) for entity categories
      - Per-document binary category labels (which of 41 categories are present)
      - Per-document category span texts (for binary categories)

    Returns:
        {doc_title: {
            "context": full_text,
            "entity_spans": [(start, end, EntityType), ...],
            "category_labels": [0/1 × 41],
            "category_spans": {cat_name: [(start, end, text), ...]},
        }}
    """
    with open(cuad_json_path, "r", encoding="utf-8") as f:
        data = json.load(f)

    result = {}
    for doc in data["data"]:
        title = doc["title"].strip()
        # CUAD has 1 paragraph per doc (the full contract text)
        para = doc["paragraphs"][0]
        context = para["context"]

        entity_spans: List[Tuple[int, int, EntityType]] = []
        category_labels = [0] * NUM_CUAD_CATEGORIES
        category_spans: Dict[str, List[Tuple[int, int, str]]] = {}

        for qa in para["qas"]:
            cat_name = _extract_category_from_question(qa["question"])
            if not cat_name:
                continue

            has_answer = not qa.get("is_impossible", False) and qa.get("answers")

            # Set category label
            if cat_name in CUAD_CATEGORY2ID:
                if has_answer:
                    category_labels[CUAD_CATEGORY2ID[cat_name]] = 1

            if not has_answer:
                continue

            spans_for_cat = []
            for ans in qa["answers"]:
                a_start = ans["answer_start"]
                a_text = ans["text"]
                a_end = a_start + len(a_text)
                spans_for_cat.append((a_start, a_end, a_text))

                # If this is an entity-bearing category, record the span
                if cat_name in _CUAD_SPAN_ENTITY_MAP:
                    etype = _CUAD_SPAN_ENTITY_MAP[cat_name]
                    entity_spans.append((a_start, a_end, etype))

            category_spans[cat_name] = spans_for_cat

        # Sort entity spans and remove duplicates
        entity_spans = sorted(set(entity_spans), key=lambda x: (x[0], -(x[1] - x[0])))
        # Remove overlapping spans (longer wins)
        filtered = []
        last_end = -1
        for s, e, et in entity_spans:
            if s >= last_end:
                filtered.append((s, e, et))
                last_end = e
        entity_spans = filtered

        result[title] = {
            "context": context,
            "entity_spans": entity_spans,
            "category_labels": category_labels,
            "category_spans": category_spans,
        }

    log.info(f"Loaded CUAD gold annotations for {len(result)} documents")
    return result


def load_cuad_master_clauses(
    csv_path: str | Path,
) -> Dict[str, List[int]]:
    """
    Load master_clauses.csv and extract per-contract category presence.

    Returns:
        {filename: [0/1 × 41]}  where filename matches CUAD_v1.json titles + .pdf
    """
    result = {}
    with open(csv_path, "r", encoding="utf-8") as f:
        reader = csv.reader(f)
        header = next(reader)

        # Answer columns are at odd indices (1, 3, 5, ..., 81)
        # The category names are the even-index columns without "-Answer" suffix
        # We map: col0=Filename, then pairs (col_i=text, col_i+1=answer)
        for row in reader:
            filename = row[0]
            labels = [0] * NUM_CUAD_CATEGORIES

            # Iterate through the 41 categories (columns 1-82, pairs of 2)
            for cat_idx, cat_name in enumerate(CUAD_CATEGORIES):
                # Answer column index: each category has text + answer columns
                # Column layout: Filename, DocName, DocName-Answer, Parties, Parties-Answer, ...
                # Category i (0-indexed) → answer column = 2 + i*2
                ans_col = 2 + cat_idx * 2
                if ans_col < len(row):
                    ans = row[ans_col].strip()
                    if ans and ans != "No" and ans != "[]":
                        labels[cat_idx] = 1

            result[filename] = labels

    log.info(f"Loaded master_clauses.csv: {len(result)} contracts")
    return result


# ── LEDGAR Loader ─────────────────────────────────────────────────────────────

@dataclass
class LEDGARProvision:
    text: str
    labels: List[str]
    cuad_category: Optional[str]  # mapped CUAD category or None
    is_immutable: bool


def load_ledgar_provisions(
    jsonl_path: str | Path,
    max_records: int = 500000,
    min_text_length: int = 20,
) -> List[LEDGARProvision]:
    """
    Load LEDGAR provisions and map their labels to CUAD categories.
    Only keeps provisions whose label maps to a known CUAD category.

    Args:
        jsonl_path: path to sec_corpus_2016-2019.jsonl
        max_records: max records to read (LEDGAR is 1.4 GB)
        min_text_length: skip very short provisions
    """
    provisions = []
    with open(jsonl_path, "r", encoding="utf-8") as f:
        for i, line in enumerate(f):
            if i >= max_records:
                break
            obj = json.loads(line)
            text = obj.get("provision", "")
            labels = obj.get("label", [])

            if len(text) < min_text_length:
                continue

            # Map LEDGAR labels to CUAD categories
            cuad_cat = None
            is_immutable = False
            for lab in labels:
                lab_lower = lab.lower().strip()
                if lab_lower in LEDGAR_TO_CUAD:
                    mapped = LEDGAR_TO_CUAD[lab_lower]
                    if mapped == "_IMMUTABLE":
                        is_immutable = True
                    else:
                        cuad_cat = mapped
                        break

            # Only keep provisions with either a CUAD mapping or immutability flag
            if cuad_cat is not None or is_immutable:
                provisions.append(LEDGARProvision(
                    text=text,
                    labels=labels,
                    cuad_category=cuad_cat,
                    is_immutable=is_immutable,
                ))

    log.info(
        f"Loaded {len(provisions)} LEDGAR provisions "
        f"(from {min(i+1, max_records)} records)"
    )
    return provisions


# ── Silver-label generation (regex fallback for MONEY / LEGAL_REF) ───────────

def extract_entity_spans_regex(
    text: str,
) -> List[Tuple[int, int, EntityType]]:
    """
    Apply regex patterns to find entity spans in text.
    Used as a fallback for MONEY and LEGAL_REF entities which are NOT
    annotated as gold spans in CUAD v1.
    Returns list of (start, end, entity_type) sorted by start position.
    """
    all_spans: List[Tuple[int, int, EntityType]] = []

    for etype, patterns in ENTITY_REGEX_MAP.items():
        for pat in patterns:
            for m in pat.finditer(text):
                all_spans.append((m.start(), m.end(), etype))

    all_spans.sort(key=lambda x: (x[0], -(x[1] - x[0])))

    filtered = []
    last_end = -1
    for start, end, etype in all_spans:
        if start >= last_end:
            filtered.append((start, end, etype))
            last_end = end

    return filtered


def merge_gold_and_regex_spans(
    gold_spans: List[Tuple[int, int, EntityType]],
    text: str,
) -> List[Tuple[int, int, EntityType]]:
    """
    Merge CUAD gold entity spans with regex-detected MONEY / LEGAL_REF spans.
    Gold spans take priority over regex when there is overlap.
    """
    # Regex only for types NOT covered by CUAD gold annotations
    regex_spans = extract_entity_spans_regex(text)
    # Only keep MONEY and LEGAL_REF from regex (PARTY/DATE/DURATION/JURISDICTION come from gold)
    regex_only = [(s, e, et) for s, e, et in regex_spans
                  if et in (EntityType.MONEY, EntityType.LEGAL_REF)]

    combined = list(gold_spans) + regex_only
    combined.sort(key=lambda x: (x[0], -(x[1] - x[0])))

    filtered = []
    last_end = -1
    for s, e, et in combined:
        if s >= last_end:
            filtered.append((s, e, et))
            last_end = e
    return filtered


def generate_obligation_labels(text: str) -> List[int]:
    """
    Generate multi-label obligation vector for text using keyword matching.
    Returns list of length NUM_OBLIGATION_LABELS (binary).
    """
    labels = [0] * NUM_OBLIGATION_LABELS
    text_lower = text.lower()

    for obligation_type, keywords in OBLIGATION_KEYWORDS.items():
        label_id = OBLIGATION_LABEL2ID[obligation_type]
        for kw in keywords:
            if kw.lower() in text_lower:
                labels[label_id] = 1
                break

    return labels


def find_obligation_trigger(text: str, obligation_type: str) -> str:
    """Find the keyword that triggered the obligation classification."""
    text_lower = text.lower()
    keywords = OBLIGATION_KEYWORDS.get(obligation_type, [])
    for kw in keywords:
        if kw.lower() in text_lower:
            return kw
    return ""


# ── Token-level BIO label alignment ──────────────────────────────────────────

def align_labels_to_tokens(
    text: str,
    entity_spans: List[Tuple[int, int, EntityType]],
    tokenizer: PreTrainedTokenizerBase,
    max_length: int = 256,
) -> Tuple[torch.Tensor, torch.Tensor, torch.Tensor]:
    """
    Tokenise `text` and produce aligned BIO NER labels.

    Returns:
        input_ids      : [max_length]
        attention_mask  : [max_length]
        ner_labels     : [max_length]  (-100 for special tokens / padding)
    """
    encoding = tokenizer(
        text,
        max_length=max_length,
        padding="max_length",
        truncation=True,
        return_offsets_mapping=True,
        return_tensors="pt",
    )

    input_ids = encoding["input_ids"].squeeze(0)
    attention_mask = encoding["attention_mask"].squeeze(0)
    offsets = encoding["offset_mapping"].squeeze(0)  # [L, 2]

    ner_labels = torch.full((max_length,), fill_value=-100, dtype=torch.long)

    for token_idx in range(max_length):
        tok_start, tok_end = offsets[token_idx].tolist()
        if tok_start == 0 and tok_end == 0:
            continue

        ner_labels[token_idx] = NER_LABEL2ID["O"]

        for e_start, e_end, etype in entity_spans:
            if tok_start >= e_start and tok_end <= e_end:
                if tok_start == e_start or (
                    token_idx > 0
                    and ner_labels[token_idx - 1].item() == NER_LABEL2ID["O"]
                ):
                    ner_labels[token_idx] = NER_LABEL2ID[f"B-{etype.value}"]
                else:
                    ner_labels[token_idx] = NER_LABEL2ID[f"I-{etype.value}"]
                break

    return input_ids, attention_mask, ner_labels


# ── CUAD Gold NER Dataset ────────────────────────────────────────────────────
# Uses gold character-level spans from CUAD_v1.json for PARTY, DATE, DURATION,
# JURISDICTION. Regex supplements MONEY and LEGAL_REF.

@dataclass
class NERSample:
    doc_id: str
    clause_idx: int
    input_ids: torch.Tensor       # [L]
    attention_mask: torch.Tensor  # [L]
    ner_labels: torch.Tensor      # [L]


class ConstraintNERDataset(Dataset):
    """
    Token-level NER dataset.
    Primary source: CUAD v1 gold spans for PARTY/DATE/DURATION/JURISDICTION.
    Secondary source: regex patterns for MONEY/LEGAL_REF.
    """

    def __init__(self, samples: List[NERSample]):
        self.samples = samples

    def __len__(self) -> int:
        return len(self.samples)

    def __getitem__(self, idx: int) -> Dict[str, torch.Tensor]:
        s = self.samples[idx]
        return {
            "input_ids": s.input_ids,
            "attention_mask": s.attention_mask,
            "labels": s.ner_labels,
        }

    @classmethod
    def from_cuad_json(
        cls,
        cuad_json_path: str | Path,
        segmented_json_path: str | Path,
        tokenizer: PreTrainedTokenizerBase,
        max_length: int = 256,
    ) -> "ConstraintNERDataset":
        """
        Build NER dataset from CUAD v1 gold annotations + segmented clauses.

        Args:
            cuad_json_path: path to CUAD_v1.json (SQuAD format with gold spans)
            segmented_json_path: path to segmented_contracts3.json (Stage 1 output)
            tokenizer: HuggingFace tokenizer for Legal-BERT
            max_length: max tokens per clause sample
        """
        # Load CUAD gold annotations (entity spans + category labels)
        cuad_data = load_cuad_gold_annotations(cuad_json_path)

        # Load segmented contracts from Stage 1 (JSON array or JSONL)
        segments = load_segments(segmented_json_path)

        samples = []
        for doc in segments:
            doc_id = doc.get("doc_id", doc.get("filename", ""))
            full_text = doc.get("text", doc.get("full_text", ""))

            # Try to match this doc to CUAD annotations by title
            gold = cuad_data.get(doc_id)

            clauses = doc.get("clauses", [])
            for ci, clause_info in enumerate(clauses):
                # Handle both formats: [start, end] or {"start": .., "end": .., "text": ..}
                if isinstance(clause_info, list):
                    cs, ce = clause_info[0], clause_info[1]
                    clause_text = full_text[cs:ce]
                elif isinstance(clause_info, dict):
                    cs = clause_info.get("start", 0)
                    ce = clause_info.get("end", 0)
                    clause_text = clause_info.get("text", full_text[cs:ce])
                else:
                    continue

                if not clause_text.strip():
                    continue

                # Build entity spans for this clause
                clause_spans: List[Tuple[int, int, EntityType]] = []

                if gold is not None:
                    # Map gold document-level spans to clause-local offsets
                    for g_start, g_end, g_etype in gold["entity_spans"]:
                        # Check if gold span overlaps with this clause
                        if g_start >= cs and g_end <= ce:
                            local_start = g_start - cs
                            local_end = g_end - cs
                            clause_spans.append((local_start, local_end, g_etype))

                # Merge with regex for MONEY and LEGAL_REF
                all_spans = merge_gold_and_regex_spans(clause_spans, clause_text)

                input_ids, attn_mask, ner_labels = align_labels_to_tokens(
                    clause_text, all_spans, tokenizer, max_length
                )
                samples.append(NERSample(
                    doc_id=doc_id,
                    clause_idx=ci,
                    input_ids=input_ids,
                    attention_mask=attn_mask,
                    ner_labels=ner_labels,
                ))

        log.info(f"Built {len(samples)} NER samples from CUAD gold + regex")
        return cls(samples)

    @classmethod
    def from_texts_regex_only(
        cls,
        texts: List[Tuple[str, str]],
        tokenizer: PreTrainedTokenizerBase,
        max_length: int = 256,
    ) -> "ConstraintNERDataset":
        """
        Fallback: build NER dataset from plain texts using regex only.
        texts: [(doc_id, clause_text), ...]
        """
        samples = []
        for doc_id, clause_text in texts:
            if not clause_text.strip():
                continue
            entity_spans = extract_entity_spans_regex(clause_text)
            input_ids, attn_mask, ner_labels = align_labels_to_tokens(
                clause_text, entity_spans, tokenizer, max_length
            )
            samples.append(NERSample(
                doc_id=doc_id,
                clause_idx=0,
                input_ids=input_ids,
                attention_mask=attn_mask,
                ner_labels=ner_labels,
            ))
        log.info(f"Built {len(samples)} NER samples (regex only)")
        return cls(samples)


# ── Obligation Dataset ────────────────────────────────────────────────────────

@dataclass
class ObligationSample:
    doc_id: str
    sentence_idx: int
    input_ids: torch.Tensor        # [L]
    attention_mask: torch.Tensor   # [L]
    obligation_labels: torch.Tensor  # [NUM_OBLIGATION_LABELS]


class ObligationDataset(Dataset):
    """
    Sentence-level obligation classification.
    Each sample is one sentence with multi-label obligation type.
    """

    def __init__(self, samples: List[ObligationSample]):
        self.samples = samples

    def __len__(self) -> int:
        return len(self.samples)

    def __getitem__(self, idx: int) -> Dict[str, torch.Tensor]:
        s = self.samples[idx]
        return {
            "input_ids": s.input_ids,
            "attention_mask": s.attention_mask,
            "labels": s.obligation_labels,
        }

    @classmethod
    def from_segmented_json(
        cls,
        segmented_json_path: str | Path,
        tokenizer: PreTrainedTokenizerBase,
        max_length: int = 128,
    ) -> "ObligationDataset":
        """
        Load from segmented contracts JSON/JSONL, split clauses into sentences,
        and label obligations via keyword matching.
        """
        segments = load_segments(segmented_json_path)

        samples = []
        for doc in segments:
            doc_id = doc.get("doc_id", doc.get("filename", ""))
            full_text = doc.get("text", doc.get("full_text", ""))
            clauses = doc.get("clauses", [])

            for ci, clause_info in enumerate(clauses):
                if isinstance(clause_info, list):
                    clause_text = full_text[clause_info[0]:clause_info[1]]
                elif isinstance(clause_info, dict):
                    clause_text = clause_info.get("text", "")
                else:
                    continue

                if not clause_text.strip():
                    continue

                # Split clause into sentences (simple split on period+space)
                sentences = [s.strip() for s in re.split(r'(?<=[.;])\s+', clause_text) if s.strip()]

                for si, sent in enumerate(sentences):
                    if len(sent) < 10:
                        continue
                    obl_labels = generate_obligation_labels(sent)

                    enc = tokenizer(
                        sent,
                        max_length=max_length,
                        padding="max_length",
                        truncation=True,
                        return_tensors="pt",
                    )
                    samples.append(ObligationSample(
                        doc_id=doc_id,
                        sentence_idx=si,
                        input_ids=enc["input_ids"].squeeze(0),
                        attention_mask=enc["attention_mask"].squeeze(0),
                        obligation_labels=torch.tensor(obl_labels, dtype=torch.float),
                    ))

        log.info(f"Loaded {len(samples)} obligation samples")
        return cls(samples)


# ── CUAD Category Classification Dataset ─────────────────────────────────────

@dataclass
class CUADCategorySample:
    doc_id: str
    clause_idx: int
    input_ids: torch.Tensor         # [L]
    attention_mask: torch.Tensor    # [L]
    category_labels: torch.Tensor   # [NUM_CUAD_CATEGORIES=41]


class CUADCategoryDataset(Dataset):
    """
    Clause-level multi-label classification across 41 CUAD categories.
    Uses gold category labels from CUAD_v1.json (which of the 41 QAs
    have answer spans overlapping this clause).
    """

    def __init__(self, samples: List[CUADCategorySample]):
        self.samples = samples

    def __len__(self) -> int:
        return len(self.samples)

    def __getitem__(self, idx: int) -> Dict[str, torch.Tensor]:
        s = self.samples[idx]
        return {
            "input_ids": s.input_ids,
            "attention_mask": s.attention_mask,
            "labels": s.category_labels,
        }

    @classmethod
    def from_cuad_json(
        cls,
        cuad_json_path: str | Path,
        segmented_json_path: str | Path,
        tokenizer: PreTrainedTokenizerBase,
        max_length: int = 256,
    ) -> "CUADCategoryDataset":
        """
        Build per-clause category labels: a clause is labeled positive for
        a CUAD category if any of that category's gold spans fall within it.
        """
        cuad_data = load_cuad_gold_annotations(cuad_json_path)

        segments = load_segments(segmented_json_path)

        samples = []
        for doc in segments:
            doc_id = doc.get("doc_id", doc.get("filename", ""))
            full_text = doc.get("text", doc.get("full_text", ""))
            gold = cuad_data.get(doc_id)
            clauses = doc.get("clauses", [])

            for ci, clause_info in enumerate(clauses):
                if isinstance(clause_info, list):
                    cs, ce = clause_info[0], clause_info[1]
                    clause_text = full_text[cs:ce]
                elif isinstance(clause_info, dict):
                    cs = clause_info.get("start", 0)
                    ce = clause_info.get("end", 0)
                    clause_text = clause_info.get("text", full_text[cs:ce])
                else:
                    continue

                if not clause_text.strip():
                    continue

                # Build clause-level category vector from gold category spans
                cat_labels = [0] * NUM_CUAD_CATEGORIES
                if gold is not None:
                    for cat_name, spans in gold["category_spans"].items():
                        if cat_name in CUAD_CATEGORY2ID:
                            # Check if any span of this category overlaps the clause
                            for sp_start, sp_end, _ in spans:
                                if sp_start < ce and sp_end > cs:
                                    cat_labels[CUAD_CATEGORY2ID[cat_name]] = 1
                                    break

                enc = tokenizer(
                    clause_text,
                    max_length=max_length,
                    padding="max_length",
                    truncation=True,
                    return_tensors="pt",
                )
                samples.append(CUADCategorySample(
                    doc_id=doc_id,
                    clause_idx=ci,
                    input_ids=enc["input_ids"].squeeze(0),
                    attention_mask=enc["attention_mask"].squeeze(0),
                    category_labels=torch.tensor(cat_labels, dtype=torch.float),
                ))

        log.info(f"Built {len(samples)} CUAD category samples")
        return cls(samples)


# ── LEDGAR Provision Classification Dataset ──────────────────────────────────

class LEDGARDataset(Dataset):
    """
    LEDGAR provision → CUAD category mapping dataset for pre-training / augmentation.
    Each sample is a LEDGAR text with its mapped CUAD category as a single-label target.
    """

    def __init__(
        self,
        provisions: List[LEDGARProvision],
        tokenizer: PreTrainedTokenizerBase,
        max_length: int = 256,
    ):
        self.provisions = provisions
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self) -> int:
        return len(self.provisions)

    def __getitem__(self, idx: int) -> Dict[str, torch.Tensor]:
        prov = self.provisions[idx]
        enc = self.tokenizer(
            prov.text,
            max_length=self.max_length,
            padding="max_length",
            truncation=True,
            return_tensors="pt",
        )

        # Multi-label: set the mapped CUAD category + obligation label
        cat_labels = [0] * NUM_CUAD_CATEGORIES
        if prov.cuad_category and prov.cuad_category in CUAD_CATEGORY2ID:
            cat_labels[CUAD_CATEGORY2ID[prov.cuad_category]] = 1

        return {
            "input_ids": enc["input_ids"].squeeze(0),
            "attention_mask": enc["attention_mask"].squeeze(0),
            "labels": torch.tensor(cat_labels, dtype=torch.float),
        }


# ── Joint Dataset (NER + Obligation + CUAD Category) ─────────────────────────

@dataclass
class JointSample:
    doc_id: str
    clause_idx: int
    input_ids: torch.Tensor          # [L]
    attention_mask: torch.Tensor     # [L]
    ner_labels: torch.Tensor         # [L]
    obligation_labels: torch.Tensor  # [NUM_OBLIGATION_LABELS=5]
    category_labels: torch.Tensor    # [NUM_CUAD_CATEGORIES=41]


class JointConstraintDataset(Dataset):
    """
    Combined dataset for joint NER + obligation + CUAD category training.
    NER: CUAD gold spans (PARTY/DATE/DURATION/JURISDICTION) + regex (MONEY/LEGAL_REF).
    Obligation: keyword-based deontic classification (5 types).
    Category: gold 41-way multi-label from CUAD_v1.json annotations.
    """

    def __init__(self, samples: List[JointSample]):
        self.samples = samples

    def __len__(self) -> int:
        return len(self.samples)

    def __getitem__(self, idx: int) -> Dict[str, torch.Tensor]:
        s = self.samples[idx]
        return {
            "input_ids": s.input_ids,
            "attention_mask": s.attention_mask,
            "ner_labels": s.ner_labels,
            "obligation_labels": s.obligation_labels,
            "category_labels": s.category_labels,
        }

    @classmethod
    def from_cuad_json(
        cls,
        cuad_json_path: str | Path,
        segmented_json_path: str | Path,
        tokenizer: PreTrainedTokenizerBase,
        max_length: int = 256,
    ) -> "JointConstraintDataset":
        """
        Build joint dataset from CUAD v1 gold annotations + segmented clauses.
        """
        cuad_data = load_cuad_gold_annotations(cuad_json_path)

        segments = load_segments(segmented_json_path)

        samples = []
        for doc in segments:
            doc_id = doc.get("doc_id", doc.get("filename", ""))
            full_text = doc.get("text", doc.get("full_text", ""))
            gold = cuad_data.get(doc_id)
            clauses = doc.get("clauses", [])

            for ci, clause_info in enumerate(clauses):
                if isinstance(clause_info, list):
                    cs, ce = clause_info[0], clause_info[1]
                    clause_text = full_text[cs:ce]
                elif isinstance(clause_info, dict):
                    cs = clause_info.get("start", 0)
                    ce = clause_info.get("end", 0)
                    clause_text = clause_info.get("text", full_text[cs:ce])
                else:
                    continue

                if not clause_text.strip():
                    continue

                # ── NER spans ──
                clause_gold_spans: List[Tuple[int, int, EntityType]] = []
                if gold is not None:
                    for g_start, g_end, g_etype in gold["entity_spans"]:
                        if g_start >= cs and g_end <= ce:
                            clause_gold_spans.append((g_start - cs, g_end - cs, g_etype))

                all_spans = merge_gold_and_regex_spans(clause_gold_spans, clause_text)
                input_ids, attn_mask, ner_labels = align_labels_to_tokens(
                    clause_text, all_spans, tokenizer, max_length
                )

                # ── Obligation labels ──
                obl_labels = generate_obligation_labels(clause_text)

                # ── CUAD category labels ──
                cat_labels = [0] * NUM_CUAD_CATEGORIES
                if gold is not None:
                    for cat_name, spans in gold["category_spans"].items():
                        if cat_name in CUAD_CATEGORY2ID:
                            for sp_start, sp_end, _ in spans:
                                if sp_start < ce and sp_end > cs:
                                    cat_labels[CUAD_CATEGORY2ID[cat_name]] = 1
                                    break

                samples.append(JointSample(
                    doc_id=doc_id,
                    clause_idx=ci,
                    input_ids=input_ids,
                    attention_mask=attn_mask,
                    ner_labels=ner_labels,
                    obligation_labels=torch.tensor(obl_labels, dtype=torch.float),
                    category_labels=torch.tensor(cat_labels, dtype=torch.float),
                ))

        log.info(f"Built {len(samples)} joint samples from CUAD gold")
        return cls(samples)


# ── Collate functions ─────────────────────────────────────────────────────────

def collate_ner(batch: List[Dict[str, torch.Tensor]]) -> Dict[str, torch.Tensor]:
    return {
        "input_ids": torch.stack([b["input_ids"] for b in batch]),
        "attention_mask": torch.stack([b["attention_mask"] for b in batch]),
        "labels": torch.stack([b["labels"] for b in batch]),
    }


def collate_obligation(batch: List[Dict[str, torch.Tensor]]) -> Dict[str, torch.Tensor]:
    return {
        "input_ids": torch.stack([b["input_ids"] for b in batch]),
        "attention_mask": torch.stack([b["attention_mask"] for b in batch]),
        "labels": torch.stack([b["labels"] for b in batch]),
    }


def collate_category(batch: List[Dict[str, torch.Tensor]]) -> Dict[str, torch.Tensor]:
    return {
        "input_ids": torch.stack([b["input_ids"] for b in batch]),
        "attention_mask": torch.stack([b["attention_mask"] for b in batch]),
        "labels": torch.stack([b["labels"] for b in batch]),
    }


def collate_joint(batch: List[Dict[str, torch.Tensor]]) -> Dict[str, torch.Tensor]:
    return {
        "input_ids": torch.stack([b["input_ids"] for b in batch]),
        "attention_mask": torch.stack([b["attention_mask"] for b in batch]),
        "ner_labels": torch.stack([b["ner_labels"] for b in batch]),
        "obligation_labels": torch.stack([b["obligation_labels"] for b in batch]),
        "category_labels": torch.stack([b["category_labels"] for b in batch]),
    }


## 4. Evaluation Metrics
- Entity-level NER P/R/F1 (exact + partial match)
- Multi-label obligation metrics (Hamming loss, per-type F1)
- 41-way CUAD category metrics (micro/macro F1)

In [ ]:
"""
constraint_evaluate.py
----------------------
Evaluation metrics for Stage 2: Constraint Extraction.

NER Metrics:
  - Entity-level Precision / Recall / F1 (exact + partial match)
  - Per-entity-type breakdown (PARTY, DATE, MONEY, DURATION, JURISDICTION, LEGAL_REF)

Obligation Metrics:
  - Multi-label accuracy, per-type P/R/F1
  - Hamming loss

CUAD Category Metrics:
  - 41-way multi-label P/R/F1 (micro, macro)
  - Per-category breakdown
"""

import logging
from collections import Counter, defaultdict
from typing import Dict, List, Optional, Tuple

# constraint_types already defined above

log = logging.getLogger(__name__)


# ── NER evaluation (entity-level) ────────────────────────────────────────────

def _decode_bio_to_spans(
    labels: List[int],
) -> List[Tuple[str, int, int]]:
    """
    Convert a BIO label sequence to entity spans.
    Returns list of (entity_type, start_token, end_token).
    """
    spans = []
    current_type = None
    current_start = None

    for i, label_id in enumerate(labels):
        if label_id == -100:
            continue
        label = NER_ID2LABEL.get(label_id, "O")

        if label.startswith("B-"):
            if current_type is not None:
                spans.append((current_type, current_start, i))
            current_type = label[2:]
            current_start = i
        elif label.startswith("I-") and current_type == label[2:]:
            continue  # extend current span
        else:
            if current_type is not None:
                spans.append((current_type, current_start, i))
                current_type = None

    if current_type is not None:
        spans.append((current_type, current_start, len(labels)))

    return spans


def compute_ner_metrics(
    all_preds: List[List[int]],
    all_labels: List[List[int]],
    partial_match: bool = True,
) -> Dict[str, float]:
    """
    Compute entity-level NER metrics.

    Args:
        all_preds  : list of per-sample predicted label sequences
        all_labels : list of per-sample ground-truth label sequences
        partial_match: if True, also compute partial match metrics
                       (50% overlap counts as half a TP)
    """
    per_type_tp: Dict[str, float] = defaultdict(float)
    per_type_fp: Dict[str, float] = defaultdict(float)
    per_type_fn: Dict[str, float] = defaultdict(float)

    for preds, labels in zip(all_preds, all_labels):
        # Strip padding
        valid_preds = [p for p, l in zip(preds, labels) if l != -100]
        valid_labels = [l for l in labels if l != -100]

        pred_spans = set(_decode_bio_to_spans(valid_preds))
        gold_spans = set(_decode_bio_to_spans(valid_labels))

        # Exact match
        for etype, ps, pe in pred_spans:
            if (etype, ps, pe) in gold_spans:
                per_type_tp[etype] += 1.0
            else:
                # Check partial match
                matched = False
                if partial_match:
                    for gt, gs, ge in gold_spans:
                        if etype == gt:
                            overlap = max(0, min(pe, ge) - max(ps, gs))
                            union = max(pe, ge) - min(ps, gs)
                            if union > 0 and overlap / union >= 0.5:
                                per_type_tp[etype] += 0.5
                                matched = True
                                break
                if not matched:
                    per_type_fp[etype] += 1.0

        for etype, gs, ge in gold_spans:
            if (etype, gs, ge) not in pred_spans:
                # Check if partially matched above
                matched = False
                if partial_match:
                    for pt, ps, pe in pred_spans:
                        if etype == pt:
                            overlap = max(0, min(pe, ge) - max(ps, gs))
                            union = max(pe, ge) - min(ps, gs)
                            if union > 0 and overlap / union >= 0.5:
                                matched = True
                                break
                if not matched:
                    per_type_fn[etype] += 1.0

    # Aggregate metrics
    all_types = set(per_type_tp) | set(per_type_fp) | set(per_type_fn)
    metrics: Dict[str, float] = {}

    total_tp = total_fp = total_fn = 0.0

    for etype in sorted(all_types):
        tp = per_type_tp[etype]
        fp = per_type_fp[etype]
        fn = per_type_fn[etype]
        total_tp += tp
        total_fp += fp
        total_fn += fn

        p = tp / (tp + fp + 1e-8)
        r = tp / (tp + fn + 1e-8)
        f1 = 2 * p * r / (p + r + 1e-8)
        metrics[f"ner_{etype}_precision"] = round(p, 4)
        metrics[f"ner_{etype}_recall"] = round(r, 4)
        metrics[f"ner_{etype}_f1"] = round(f1, 4)

    # Micro-averaged
    p = total_tp / (total_tp + total_fp + 1e-8)
    r = total_tp / (total_tp + total_fn + 1e-8)
    f1 = 2 * p * r / (p + r + 1e-8)
    metrics["ner_precision"] = round(p, 4)
    metrics["ner_recall"] = round(r, 4)
    metrics["ner_f1"] = round(f1, 4)

    return metrics


# ── Token-level NER metrics (for comparison) ─────────────────────────────────

def compute_token_level_metrics(
    all_preds: List[List[int]],
    all_labels: List[List[int]],
) -> Dict[str, float]:
    """Simple token-level accuracy and per-label P/R/F1."""
    correct = total = 0
    per_label_tp: Dict[int, int] = defaultdict(int)
    per_label_fp: Dict[int, int] = defaultdict(int)
    per_label_fn: Dict[int, int] = defaultdict(int)

    for preds, labels in zip(all_preds, all_labels):
        for p, l in zip(preds, labels):
            if l == -100:
                continue
            total += 1
            if p == l:
                correct += 1
                per_label_tp[l] += 1
            else:
                per_label_fp[p] += 1
                per_label_fn[l] += 1

    metrics = {"token_accuracy": round(correct / max(total, 1), 4)}

    # Per non-O label
    for label_id in range(1, NUM_NER_LABELS):
        tp = per_label_tp[label_id]
        fp = per_label_fp[label_id]
        fn = per_label_fn[label_id]
        p = tp / (tp + fp + 1e-8)
        r = tp / (tp + fn + 1e-8)
        f1 = 2 * p * r / (p + r + 1e-8)
        label_name = NER_ID2LABEL.get(label_id, str(label_id))
        metrics[f"token_{label_name}_f1"] = round(f1, 4)

    return metrics


# ── Obligation evaluation ─────────────────────────────────────────────────────

def compute_obligation_metrics(
    all_preds: List[List[int]],     # list of multi-label binary vectors
    all_labels: List[List[int]],
) -> Dict[str, float]:
    """
    Multi-label obligation classification metrics.

    Args:
        all_preds  : list of [NUM_OBLIGATION_LABELS] binary vectors
        all_labels : list of [NUM_OBLIGATION_LABELS] binary vectors
    """
    n = len(all_preds)
    if n == 0:
        return {}

    per_type_tp: Dict[int, int] = defaultdict(int)
    per_type_fp: Dict[int, int] = defaultdict(int)
    per_type_fn: Dict[int, int] = defaultdict(int)

    hamming_errors = 0
    total_labels = 0
    exact_match = 0

    for preds, labels in zip(all_preds, all_labels):
        exact = True
        for i in range(NUM_OBLIGATION_LABELS):
            p = preds[i]
            l = labels[i]
            total_labels += 1

            if p == 1 and l == 1:
                per_type_tp[i] += 1
            elif p == 1 and l == 0:
                per_type_fp[i] += 1
                hamming_errors += 1
                exact = False
            elif p == 0 and l == 1:
                per_type_fn[i] += 1
                hamming_errors += 1
                exact = False

        if exact:
            exact_match += 1

    metrics: Dict[str, float] = {
        "obligation_exact_match": round(exact_match / max(n, 1), 4),
        "obligation_hamming_loss": round(hamming_errors / max(total_labels, 1), 4),
    }

    total_tp = total_fp = total_fn = 0

    for i in range(NUM_OBLIGATION_LABELS):
        tp = per_type_tp[i]
        fp = per_type_fp[i]
        fn = per_type_fn[i]
        total_tp += tp
        total_fp += fp
        total_fn += fn

        p = tp / (tp + fp + 1e-8)
        r = tp / (tp + fn + 1e-8)
        f1 = 2 * p * r / (p + r + 1e-8)
        label_name = OBLIGATION_ID2LABEL[i]
        metrics[f"obligation_{label_name}_precision"] = round(p, 4)
        metrics[f"obligation_{label_name}_recall"] = round(r, 4)
        metrics[f"obligation_{label_name}_f1"] = round(f1, 4)

    # Micro-averaged
    p = total_tp / (total_tp + total_fp + 1e-8)
    r = total_tp / (total_tp + total_fn + 1e-8)
    f1 = 2 * p * r / (p + r + 1e-8)
    metrics["obligation_micro_precision"] = round(p, 4)
    metrics["obligation_micro_recall"] = round(r, 4)
    metrics["obligation_micro_f1"] = round(f1, 4)

    return metrics


# ── CUAD Category evaluation (41-way multi-label) ────────────────────────────

def compute_category_metrics(
    all_preds: List[List[int]],
    all_labels: List[List[int]],
) -> Dict[str, float]:
    """
    Multi-label CUAD category classification metrics.

    Args:
        all_preds  : list of [NUM_CUAD_CATEGORIES] binary vectors
        all_labels : list of [NUM_CUAD_CATEGORIES] binary vectors
    """
    n = len(all_preds)
    if n == 0:
        return {}

    per_cat_tp: Dict[int, int] = defaultdict(int)
    per_cat_fp: Dict[int, int] = defaultdict(int)
    per_cat_fn: Dict[int, int] = defaultdict(int)

    hamming_errors = 0
    total_labels = 0
    exact_match = 0

    for preds, labels in zip(all_preds, all_labels):
        exact = True
        for i in range(NUM_CUAD_CATEGORIES):
            p = preds[i] if i < len(preds) else 0
            l = labels[i] if i < len(labels) else 0
            total_labels += 1

            if p == 1 and l == 1:
                per_cat_tp[i] += 1
            elif p == 1 and l == 0:
                per_cat_fp[i] += 1
                hamming_errors += 1
                exact = False
            elif p == 0 and l == 1:
                per_cat_fn[i] += 1
                hamming_errors += 1
                exact = False

        if exact:
            exact_match += 1

    metrics: Dict[str, float] = {
        "category_exact_match": round(exact_match / max(n, 1), 4),
        "category_hamming_loss": round(hamming_errors / max(total_labels, 1), 4),
    }

    # Micro-averaged
    total_tp = sum(per_cat_tp.values())
    total_fp = sum(per_cat_fp.values())
    total_fn = sum(per_cat_fn.values())

    p = total_tp / (total_tp + total_fp + 1e-8)
    r = total_tp / (total_tp + total_fn + 1e-8)
    f1 = 2 * p * r / (p + r + 1e-8)
    metrics["category_micro_precision"] = round(p, 4)
    metrics["category_micro_recall"] = round(r, 4)
    metrics["category_micro_f1"] = round(f1, 4)

    # Macro-averaged (over categories with support > 0)
    f1_scores = []
    for i in range(NUM_CUAD_CATEGORIES):
        tp = per_cat_tp[i]
        fp = per_cat_fp[i]
        fn = per_cat_fn[i]
        if tp + fp + fn == 0:
            continue
        ci_p = tp / (tp + fp + 1e-8)
        ci_r = tp / (tp + fn + 1e-8)
        ci_f1 = 2 * ci_p * ci_r / (ci_p + ci_r + 1e-8)
        f1_scores.append(ci_f1)

        cat_name = CUAD_ID2CATEGORY.get(i, str(i))
        metrics[f"category_{cat_name}_f1"] = round(ci_f1, 4)

    if f1_scores:
        metrics["category_macro_f1"] = round(sum(f1_scores) / len(f1_scores), 4)

    return metrics


# ── Combined report ───────────────────────────────────────────────────────────

def compute_all_metrics(
    ner_preds: List[List[int]],
    ner_labels: List[List[int]],
    obligation_preds: List[List[int]],
    obligation_labels: List[List[int]],
    category_preds: Optional[List[List[int]]] = None,
    category_labels: Optional[List[List[int]]] = None,
) -> Dict[str, float]:
    """Compute and merge all Stage 2 metrics."""
    metrics = {}
    metrics.update(compute_ner_metrics(ner_preds, ner_labels))
    metrics.update(compute_token_level_metrics(ner_preds, ner_labels))
    metrics.update(compute_obligation_metrics(obligation_preds, obligation_labels))
    if category_preds is not None and category_labels is not None:
        metrics.update(compute_category_metrics(category_preds, category_labels))
    return metrics


def print_metrics(metrics: Dict[str, float]):
    """Pretty-print evaluation metrics."""
    log.info("\n" + "=" * 60)
    log.info("  STAGE 2 — CONSTRAINT EXTRACTION EVALUATION")
    log.info("=" * 60)

    # NER summary
    log.info(f"\n  NER (entity-level):")
    log.info(f"    Precision: {metrics.get('ner_precision', 0):.4f}")
    log.info(f"    Recall:    {metrics.get('ner_recall', 0):.4f}")
    log.info(f"    F1:        {metrics.get('ner_f1', 0):.4f}")

    # Per entity type
    for etype in EntityType:
        key = f"ner_{etype.value}_f1"
        if key in metrics:
            log.info(f"    {etype.value:12s}  F1={metrics[key]:.4f}")

    # Obligation summary
    log.info(f"\n  Obligation Classification:")
    log.info(f"    Micro F1:      {metrics.get('obligation_micro_f1', 0):.4f}")
    log.info(f"    Exact Match:   {metrics.get('obligation_exact_match', 0):.4f}")
    log.info(f"    Hamming Loss:  {metrics.get('obligation_hamming_loss', 0):.4f}")

    for otype in ObligationType:
        key = f"obligation_{otype.value}_f1"
        if key in metrics:
            log.info(f"    {otype.value:14s}  F1={metrics[key]:.4f}")

    # CUAD Category summary
    if "category_micro_f1" in metrics:
        log.info(f"\n  CUAD Category Classification (41-way):")
        log.info(f"    Micro F1:      {metrics.get('category_micro_f1', 0):.4f}")
        log.info(f"    Macro F1:      {metrics.get('category_macro_f1', 0):.4f}")
        log.info(f"    Exact Match:   {metrics.get('category_exact_match', 0):.4f}")
        log.info(f"    Hamming Loss:  {metrics.get('category_hamming_loss', 0):.4f}")

    log.info("=" * 60)


## 5. Constraint Extractor
Hybrid extraction pipeline combining rule-based regex + neural model predictions.

In [ ]:
"""
constraint_extractor.py
-----------------------
Main Stage 2 pipeline: extracts constraints from clause segments produced
by Stage 1 (Structural Segmentation).

Combines three extraction strategies:
  1. Neural NER  — Legal-BERT token classifier for PARTY, DATE, MONEY,
                    DURATION, JURISDICTION, LEGAL_REF (13 BIO labels).
  2. Rule-based  — Regex patterns as fallback / augmentation.
  3. Obligation  — Keyword + neural classification of deontic modality.
  4. CUAD Category — 41-way multi-label clause classification.

Input:  List of clause spans (from Stage 1 ClauseSegmenter)
Output: DocumentConstraints with per-clause constraint profiles
"""

import json
import logging
import re
from dataclasses import dataclass
from pathlib import Path
from typing import Dict, List, Optional, Tuple

import torch
from transformers import AutoTokenizer

# constraint_types already defined above
# constraint_dataset already defined above
# constraint_model already defined above

log = logging.getLogger(__name__)


# ── Party name extraction (heuristic + pattern-based) ─────────────────────────

# Patterns that commonly introduce party names in contracts
_PARTY_INTRO_PATTERNS = [
    # "Acme Corp., a Delaware corporation ("the Company")"
    re.compile(
        r"(?:by\s+and\s+between|between|among)\s+"
        r"([A-Z][A-Za-z\s,\.&]+?)(?:\s*,\s*a\s+\w+\s+\w+|\s*\()",
        re.MULTILINE,
    ),
    # "XYZ Inc. (the "Licensee")"
    re.compile(
        r"([A-Z][A-Za-z\s,\.&]{2,50}?(?:Inc|Corp|LLC|Ltd|LP|LLP|Co|Company|"
        r"Corporation|Limited|Partners|Group)\.?)\s*\(",
        re.IGNORECASE,
    ),
    # Defined terms in quotes: "the "Company""
    re.compile(
        r'(?:the\s+|")(Company|Licensor|Licensee|Seller|Buyer|Lessee|'
        r"Lessor|Borrower|Lender|Client|Consultant|Contractor|"
        r'Vendor|Supplier|Distributor|Agent|Principal)"?',
        re.IGNORECASE,
    ),
]

# Suffix patterns for company names
_COMPANY_SUFFIXES = re.compile(
    r"\b(?:Inc|Corp|LLC|Ltd|LLP|LP|Co|Company|Corporation|"
    r"Limited|Partners|Group|Holdings|Enterprises|International|"
    r"Industries|Associates|Incorporated|Technologies|Solutions)\.?\b",
    re.IGNORECASE,
)


def extract_party_spans(text: str) -> List[Tuple[int, int]]:
    """
    Extract likely party name spans using heuristic patterns.
    Returns (start, end) offsets within text.
    """
    spans = []
    for pat in _PARTY_INTRO_PATTERNS:
        for m in pat.finditer(text):
            if m.lastindex:
                g = m.group(1).strip().rstrip(",.")
                if len(g) > 2 and len(g) < 100:
                    start = m.start(1)
                    end = start + len(g)
                    spans.append((start, end))

    # Also find company names by suffix
    for m in _COMPANY_SUFFIXES.finditer(text):
        # walk backwards to find start of company name (uppercase word sequence)
        name_start = m.start()
        while name_start > 0 and (text[name_start - 1].isalpha() or text[name_start - 1] in " ,.-&"):
            name_start -= 1
            if name_start > 0 and text[name_start] == "\n":
                name_start += 1
                break
        name = text[name_start : m.end()].strip()
        if len(name) > 3 and name[0].isupper():
            spans.append((name_start, m.end()))

    # Deduplicate overlapping spans
    spans.sort(key=lambda x: (x[0], -(x[1] - x[0])))
    filtered = []
    last_end = -1
    for s, e in spans:
        if s >= last_end:
            filtered.append((s, e))
            last_end = e
    return filtered


# ── Immutability detection ────────────────────────────────────────────────────

_IMMUTABLE_PATTERNS = [
    re.compile(r"^\s*definitions?\s*$", re.IGNORECASE | re.MULTILINE),
    re.compile(r'"\w+" (?:shall )?means?\b', re.IGNORECASE),
    re.compile(r"\bgoverning law\b", re.IGNORECASE),
    re.compile(r"\bjurisdiction\b", re.IGNORECASE),
    re.compile(r"\bchoice of law\b", re.IGNORECASE),
    re.compile(r"\bentire agreement\b", re.IGNORECASE),
    re.compile(r"\bseverability\b", re.IGNORECASE),
]


def is_immutable_clause(text: str) -> bool:
    """Check if this clause is a boilerplate/definition that should not be simplified."""
    return any(p.search(text) for p in _IMMUTABLE_PATTERNS)


# ── Main Extractor ────────────────────────────────────────────────────────────

class ConstraintExtractor:
    """
    Full constraint extraction pipeline for one document.

    Modes:
      - "rule_based" : regex + keyword patterns only (no GPU needed)
      - "neural"     : Legal-BERT NER + obligation classifier
      - "hybrid"     : neural + rule-based (default, best quality)
    """

    def __init__(
        self,
        mode: str = "hybrid",
        ner_model: Optional[ConstraintNERModel] = None,
        joint_model: Optional[JointConstraintModel] = None,
        obligation_model: Optional[ObligationClassifier] = None,
        category_model: Optional[CUADCategoryClassifier] = None,
        tokenizer: Optional[AutoTokenizer] = None,
        device: Optional[torch.device] = None,
        ner_max_length: int = 256,
        obligation_threshold: float = 0.5,
        category_threshold: float = 0.5,
    ):
        self.mode = mode
        self.ner_model = ner_model
        self.joint_model = joint_model
        self.obligation_model = obligation_model
        self.category_model = category_model
        self.tokenizer = tokenizer
        self.device = device or torch.device("cpu")
        self.ner_max_length = ner_max_length
        self.obligation_threshold = obligation_threshold
        self.category_threshold = category_threshold

        if mode in ("neural", "hybrid"):
            if joint_model is not None:
                joint_model.eval()
                joint_model.to(self.device)
            else:
                if ner_model is not None:
                    ner_model.eval()
                    ner_model.to(self.device)
                if obligation_model is not None:
                    obligation_model.eval()
                    obligation_model.to(self.device)

    # ── Public API ────────────────────────────────────────────────────────────

    def extract(
        self,
        doc_id: str,
        clause_spans: List[Dict],
        full_text: str,
        cuad_categories: Optional[Dict[int, List[str]]] = None,
    ) -> DocumentConstraints:
        """
        Extract constraints from a segmented document.

        Args:
            doc_id         : document identifier
            clause_spans   : list of {"index", "text", "start_char", "end_char"}
                             (output of Stage 1 inference.ClauseSegmenter)
            full_text      : full document text
            cuad_categories: optional dict mapping clause_index → list of CUAD
                             category names (from Stage 1 multitask model)
        """
        cuad_categories = cuad_categories or {}

        clause_constraints = []
        for span in clause_spans:
            cidx = span["index"]
            ctext = span["text"]
            cstart = span["start_char"]
            cend = span["end_char"]

            cats = cuad_categories.get(cidx, [])

            cc = self._extract_clause(
                clause_index=cidx,
                clause_text=ctext,
                start_char=cstart,
                end_char=cend,
                cuad_cats=cats,
            )
            clause_constraints.append(cc)

        return DocumentConstraints(doc_id=doc_id, clauses=clause_constraints)

    # ── Per-clause extraction ─────────────────────────────────────────────────

    def _extract_clause(
        self,
        clause_index: int,
        clause_text: str,
        start_char: int,
        end_char: int,
        cuad_cats: List[str],
    ) -> ClauseConstraints:
        """Extract all constraints from a single clause."""

        entities: List[ExtractedEntity] = []
        obligations: List[ObligationLabel] = []

        # ── 1. Rule-based entity extraction ───────────────────────────────────
        if self.mode in ("rule_based", "hybrid"):
            entities.extend(self._extract_entities_regex(clause_text))
            entities.extend(self._extract_parties_heuristic(clause_text))
            obligations.extend(self._extract_obligations_keywords(clause_text))

        # ── 2. Neural extraction ──────────────────────────────────────────────
        if self.mode in ("neural", "hybrid") and self.tokenizer is not None:
            neural_entities, neural_obligations, neural_cats = self._extract_neural(clause_text)
            if self.mode == "hybrid":
                entities = self._merge_entities(entities, neural_entities)
                obligations = self._merge_obligations(obligations, neural_obligations)
            else:
                entities = neural_entities
                obligations = neural_obligations
            # Merge neural categories with provided CUAD categories
            for nc in neural_cats:
                if nc not in cuad_cats:
                    cuad_cats.append(nc)

        # ── 3. CUAD category-based enrichment ─────────────────────────────────
        entities = self._enrich_from_cuad(entities, clause_text, cuad_cats)

        # ── 4. Immutability check ─────────────────────────────────────────────
        immutable = is_immutable_clause(clause_text)

        return ClauseConstraints(
            clause_index=clause_index,
            clause_text=clause_text,
            start_char=start_char,
            end_char=end_char,
            entities=entities,
            obligations=obligations,
            cuad_categories=cuad_cats,
            immutable=immutable,
        )

    # ── Rule-based extraction methods ─────────────────────────────────────────

    def _extract_entities_regex(self, text: str) -> List[ExtractedEntity]:
        """Extract entities using regex patterns."""
        spans = extract_entity_spans_regex(text)
        return [
            ExtractedEntity(
                entity_type=etype,
                text=text[start:end],
                start_char=start,
                end_char=end,
                confidence=0.8,  # rule-based confidence
            )
            for start, end, etype in spans
        ]

    def _extract_parties_heuristic(self, text: str) -> List[ExtractedEntity]:
        """Extract party names using heuristic patterns."""
        spans = extract_party_spans(text)
        return [
            ExtractedEntity(
                entity_type=EntityType.PARTY,
                text=text[start:end].strip(),
                start_char=start,
                end_char=end,
                confidence=0.7,
            )
            for start, end in spans
        ]

    def _extract_obligations_keywords(self, text: str) -> List[ObligationLabel]:
        """Classify obligation types using keyword patterns."""
        labels = generate_obligation_labels(text)
        obligations = []
        for i, val in enumerate(labels):
            if val == 1:
                obl_type = ObligationType(OBLIGATION_ID2LABEL[i])
                trigger = find_obligation_trigger(text, obl_type.value)
                obligations.append(ObligationLabel(
                    obligation_type=obl_type,
                    confidence=0.75,
                    trigger_phrase=trigger,
                ))
        return obligations

    # ── Neural extraction ─────────────────────────────────────────────────────

    @torch.no_grad()
    def _extract_neural(
        self, clause_text: str
    ) -> Tuple[List[ExtractedEntity], List[ObligationLabel], List[str]]:
        """Run neural models on a clause and decode predictions.
        Returns (entities, obligations, cuad_categories)."""
        entities: List[ExtractedEntity] = []
        obligations: List[ObligationLabel] = []
        categories: List[str] = []

        enc = self.tokenizer(
            clause_text,
            max_length=self.ner_max_length,
            padding="max_length",
            truncation=True,
            return_offsets_mapping=True,
            return_tensors="pt",
        )

        input_ids = enc["input_ids"].to(self.device)
        attention_mask = enc["attention_mask"].to(self.device)
        offsets = enc["offset_mapping"].squeeze(0)  # [L, 2]

        if self.joint_model is not None:
            ner_preds, obl_preds, cat_preds = self.joint_model.predict(
                input_ids, attention_mask,
                self.obligation_threshold,
                self.category_threshold,
            )
            ner_preds = ner_preds[0].cpu()   # [L]
            obl_preds = obl_preds[0].cpu()   # [C_obl]
            cat_preds = cat_preds[0].cpu()   # [C_cat]

            # Decode category predictions
            for i in range(cat_preds.shape[0]):
                if cat_preds[i] == 1:
                    categories.append(CUAD_ID2CATEGORY[i])
        else:
            ner_preds = None
            obl_preds = None

            if self.ner_model is not None:
                ner_preds = self.ner_model.predict(input_ids, attention_mask)[0].cpu()

            if self.obligation_model is not None:
                obl_preds = self.obligation_model.predict(
                    input_ids, attention_mask, self.obligation_threshold
                )[0].cpu()

            if self.category_model is not None:
                cat_preds = self.category_model.predict(
                    input_ids, attention_mask, self.category_threshold
                )[0].cpu()
                for i in range(cat_preds.shape[0]):
                    if cat_preds[i] == 1:
                        categories.append(CUAD_ID2CATEGORY[i])

        # Decode NER predictions into entity spans
        if ner_preds is not None:
            entities = self._decode_ner_predictions(
                clause_text, ner_preds, offsets
            )

        # Decode obligation predictions
        if obl_preds is not None:
            for i in range(obl_preds.shape[0]):
                if obl_preds[i] == 1:
                    obl_type = ObligationType(OBLIGATION_ID2LABEL[i])
                    obligations.append(ObligationLabel(
                        obligation_type=obl_type,
                        confidence=0.85,
                        trigger_phrase="[neural]",
                    ))

        return entities, obligations, categories

    def _decode_ner_predictions(
        self,
        text: str,
        ner_preds: torch.Tensor,   # [L]
        offsets: torch.Tensor,      # [L, 2]
    ) -> List[ExtractedEntity]:
        """Convert BIO prediction sequence to entity spans."""
        entities = []
        current_entity = None
        current_start = None
        current_end = None

        for tok_idx in range(ner_preds.shape[0]):
            tok_start, tok_end = offsets[tok_idx].tolist()
            if tok_start == 0 and tok_end == 0:
                continue  # special token

            label = NER_ID2LABEL.get(ner_preds[tok_idx].item(), "O")

            if label.startswith("B-"):
                # Flush previous entity
                if current_entity is not None:
                    entities.append(ExtractedEntity(
                        entity_type=EntityType(current_entity),
                        text=text[current_start:current_end].strip(),
                        start_char=current_start,
                        end_char=current_end,
                        confidence=0.85,
                    ))
                current_entity = label[2:]
                current_start = int(tok_start)
                current_end = int(tok_end)

            elif label.startswith("I-") and current_entity == label[2:]:
                current_end = int(tok_end)

            else:
                if current_entity is not None:
                    entities.append(ExtractedEntity(
                        entity_type=EntityType(current_entity),
                        text=text[current_start:current_end].strip(),
                        start_char=current_start,
                        end_char=current_end,
                        confidence=0.85,
                    ))
                    current_entity = None

        # Flush last entity
        if current_entity is not None:
            entities.append(ExtractedEntity(
                entity_type=EntityType(current_entity),
                text=text[current_start:current_end].strip(),
                start_char=current_start,
                end_char=current_end,
                confidence=0.85,
            ))

        return entities

    # ── Merging strategies ────────────────────────────────────────────────────

    @staticmethod
    def _merge_entities(
        rule_entities: List[ExtractedEntity],
        neural_entities: List[ExtractedEntity],
    ) -> List[ExtractedEntity]:
        """
        Merge rule-based and neural entities. Neural entities win on overlap.
        Non-overlapping rule-based entities are kept.
        """
        neural_ranges = [(e.start_char, e.end_char) for e in neural_entities]
        merged = list(neural_entities)

        for re_ent in rule_entities:
            overlaps = any(
                re_ent.start_char < ne and re_ent.end_char > ns
                for ns, ne in neural_ranges
            )
            if not overlaps:
                merged.append(re_ent)

        merged.sort(key=lambda e: e.start_char)
        return merged

    @staticmethod
    def _merge_obligations(
        rule_obls: List[ObligationLabel],
        neural_obls: List[ObligationLabel],
    ) -> List[ObligationLabel]:
        """Merge obligations: union of both, neural confidence takes precedence."""
        seen_types = set()
        merged = []

        for obl in neural_obls:
            merged.append(obl)
            seen_types.add(obl.obligation_type)

        for obl in rule_obls:
            if obl.obligation_type not in seen_types:
                merged.append(obl)
                seen_types.add(obl.obligation_type)

        return merged

    # ── CUAD enrichment ───────────────────────────────────────────────────────

    def _enrich_from_cuad(
        self,
        entities: List[ExtractedEntity],
        clause_text: str,
        cuad_cats: List[str],
    ) -> List[ExtractedEntity]:
        """
        If CUAD categories suggest specific entity types that weren't found
        by regex/neural, try harder with targeted patterns.
        """
        found_types = {e.entity_type for e in entities}

        for cat in cuad_cats:
            expected_types = CUAD_TO_ENTITY_TYPE.get(cat, [])
            for etype in expected_types:
                if etype not in found_types:
                    # Try regex for this specific type
                    from constraint_dataset import ENTITY_REGEX_MAP
                    patterns = ENTITY_REGEX_MAP.get(etype, [])
                    for pat in patterns:
                        for m in pat.finditer(clause_text):
                            entities.append(ExtractedEntity(
                                entity_type=etype,
                                text=m.group(),
                                start_char=m.start(),
                                end_char=m.end(),
                                confidence=0.6,
                            ))
                            found_types.add(etype)
                            break
                        if etype in found_types:
                            break

        return entities

    # ── Serialisation ─────────────────────────────────────────────────────────

    @classmethod
    def from_pretrained(
        cls,
        checkpoint_dir: str | Path,
        mode: str = "hybrid",
        device: Optional[torch.device] = None,
    ) -> "ConstraintExtractor":
        """Load a trained joint model from checkpoint."""
        checkpoint_dir = Path(checkpoint_dir)
        if device is None:
            device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

        # Load config
        with open(checkpoint_dir / "constraint_config.json") as f:
            cfg_dict = json.load(f)
        cfg = ConstraintModelConfig(**cfg_dict)

        # Load model
        joint_model = JointConstraintModel(cfg)
        state = torch.load(
            checkpoint_dir / "constraint_model.pt",
            map_location=device,
            weights_only=True,
        )
        joint_model.load_state_dict(state)

        tokenizer = AutoTokenizer.from_pretrained(str(checkpoint_dir))

        return cls(
            mode=mode,
            joint_model=joint_model,
            tokenizer=tokenizer,
            device=device,
            ner_max_length=cfg.max_seq_length,
        )

    @classmethod
    def rule_based_only(cls) -> "ConstraintExtractor":
        """Create a rule-based-only extractor (no model needed)."""
        return cls(mode="rule_based")


## 6. Inference Pipeline
End-to-end: Stage 1 segmentation → Stage 2 constraint extraction.

In [ ]:
"""
constraint_inference.py
-----------------------
End-to-end inference pipeline:
  Stage 1 (segmentation) → Stage 2 (constraint extraction)

Given raw contract text, produces a structured DocumentConstraints object
with per-clause entities and obligation types.

Usage (CLI):
  python constraint_inference.py \
    --stage1_model checkpoints/stage1/best_model \
    --stage2_model checkpoints/stage2/best_model \
    --input contract.txt \
    --output constraints.json

Usage (Python):
  from constraint_inference import ConstraintPipeline
  pipeline = ConstraintPipeline.from_pretrained(
      stage1_dir="checkpoints/stage1/best_model",
      stage2_dir="checkpoints/stage2/best_model",
  )
  result = pipeline.process("THIS AGREEMENT is entered into ...")
  print(result.to_dict())
"""

import argparse
import json
import logging
import sys
from dataclasses import asdict
from pathlib import Path
from typing import List, Optional

import torch

# ConstraintExtractor already defined above
# DocumentConstraints already defined above

log = logging.getLogger(__name__)


class ConstraintPipeline:
    """
    Full two-stage pipeline:
      Stage 1 — Segment contract text into clauses
      Stage 2 — Extract constraints from each clause
    """

    def __init__(
        self,
        stage1_segmenter=None,
        stage2_extractor: Optional[ConstraintExtractor] = None,
    ):
        self.stage1_segmenter = stage1_segmenter
        self.stage2_extractor = stage2_extractor or ConstraintExtractor.rule_based_only()

    def process(
        self,
        text: str,
        doc_id: str = "document",
    ) -> DocumentConstraints:
        """
        Run the full pipeline on raw contract text.

        Returns DocumentConstraints with per-clause entity and obligation profiles.
        """
        # ── Stage 1: Segmentation ─────────────────────────────────────────────
        if self.stage1_segmenter is not None:
            clause_spans_raw = self.stage1_segmenter.segment(text)
            clause_spans = [
                {
                    "index": cs.index,
                    "text": cs.text,
                    "start_char": cs.start_char,
                    "end_char": cs.end_char,
                }
                for cs in clause_spans_raw
            ]
        else:
            # Fallback: treat entire text as one clause
            clause_spans = [
                {
                    "index": 0,
                    "text": text,
                    "start_char": 0,
                    "end_char": len(text),
                }
            ]

        if not clause_spans:
            return DocumentConstraints(doc_id=doc_id, clauses=[])

        # ── Stage 2: Constraint extraction ────────────────────────────────────
        return self.stage2_extractor.extract(
            doc_id=doc_id,
            clause_spans=clause_spans,
            full_text=text,
        )

    def process_batch(
        self,
        texts: List[str],
        doc_ids: Optional[List[str]] = None,
    ) -> List[DocumentConstraints]:
        """Process multiple documents."""
        if doc_ids is None:
            doc_ids = [f"doc_{i}" for i in range(len(texts))]
        return [
            self.process(text, doc_id)
            for text, doc_id in zip(texts, doc_ids)
        ]

    # ── Factory methods ───────────────────────────────────────────────────────

    @classmethod
    def from_pretrained(
        cls,
        stage1_dir: Optional[str | Path] = None,
        stage2_dir: Optional[str | Path] = None,
        mode: str = "hybrid",
        device: Optional[torch.device] = None,
    ) -> "ConstraintPipeline":
        """
        Load pre-trained models for both stages.

        Args:
            stage1_dir : path to Stage 1 checkpoint (or None for no segmentation)
            stage2_dir : path to Stage 2 checkpoint (or None for rule-based only)
            mode       : "hybrid", "neural", or "rule_based"
            device     : torch device
        """
        stage1_segmenter = None
        stage2_extractor = None

        if stage1_dir is not None:
            # Import Stage 1 inference
            sys.path.insert(
                0,
                str(Path(__file__).resolve().parent.parent / "code_stage1" / "code_stage1"),
            )
            from inference import ClauseSegmenter
            stage1_segmenter = ClauseSegmenter.from_pretrained(
                stage1_dir, device=device
            )

        if stage2_dir is not None and mode != "rule_based":
            stage2_extractor = ConstraintExtractor.from_pretrained(
                stage2_dir, mode=mode, device=device
            )
        else:
            stage2_extractor = ConstraintExtractor.rule_based_only()

        return cls(
            stage1_segmenter=stage1_segmenter,
            stage2_extractor=stage2_extractor,
        )

    @classmethod
    def rule_based(
        cls,
        stage1_dir: Optional[str | Path] = None,
        device: Optional[torch.device] = None,
    ) -> "ConstraintPipeline":
        """Create pipeline with rule-based Stage 2 (no GPU needed for Stage 2)."""
        return cls.from_pretrained(
            stage1_dir=stage1_dir,
            stage2_dir=None,
            mode="rule_based",
            device=device,
        )



## 7. Training Utilities
Differential learning rate optimizer and checkpoint saving.

In [ ]:
import dataclasses

def build_optimiser(model, encoder_lr, head_lr, weight_decay):
    """Separate LR for encoder vs. task heads."""
    encoder_params = []
    head_params = []
    for name, param in model.named_parameters():
        if "encoder" in name:
            encoder_params.append(param)
        else:
            head_params.append(param)
    param_groups = [
        {"params": encoder_params, "lr": encoder_lr},
        {"params": head_params, "lr": head_lr},
    ]
    from torch.optim import AdamW
    return AdamW(param_groups, weight_decay=weight_decay, betas=(0.9, 0.999))


def save_checkpoint(model, tokenizer, cfg, path):
    from pathlib import Path as P
    p = P(path)
    p.mkdir(parents=True, exist_ok=True)
    torch.save(model.state_dict(), p / "constraint_model.pt")
    tokenizer.save_pretrained(str(p))
    with open(p / "constraint_config.json", "w") as f:
        json.dump(dataclasses.asdict(cfg), f, indent=2)
    print(f"  Checkpoint saved to {p}")

## 8. Load Data & Build Datasets
Loads CUAD gold annotations + Stage 1 JSONL segments and builds joint training/validation datasets.

In [ ]:
import torch
from transformers import AutoTokenizer
from torch.utils.data import DataLoader
from pathlib import Path

torch.manual_seed(SEED)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")

tokenizer = AutoTokenizer.from_pretrained(ENCODER_MODEL)

print("Loading training data from Stage 1 JSONL …")
train_ds = JointConstraintDataset.from_cuad_json(
    CUAD_JSON, TRAIN_JSONL, tokenizer, MAX_SEQ_LENGTH
)
print(f"  Train samples: {len(train_ds)}")

print("Loading validation data …")
val_ds = JointConstraintDataset.from_cuad_json(
    CUAD_JSON, VAL_JSONL, tokenizer, MAX_SEQ_LENGTH
)
print(f"  Val samples:   {len(val_ds)}")

train_loader = DataLoader(
    train_ds, batch_size=BATCH_SIZE, shuffle=True,
    collate_fn=collate_joint, num_workers=2, pin_memory=True,
)
val_loader = DataLoader(
    val_ds, batch_size=BATCH_SIZE, shuffle=False,
    collate_fn=collate_joint, num_workers=2,
)

print(f"\nDataLoaders ready: {len(train_loader)} train batches, {len(val_loader)} val batches")

## 9. Build Model

In [ ]:
cfg = ConstraintModelConfig(
    encoder_model_name=ENCODER_MODEL,
    freeze_encoder_epochs=FREEZE_EPOCHS,
    use_ner_crf=True,
    obligation_loss_weight=0.5,
    category_loss_weight=0.3,
    max_seq_length=MAX_SEQ_LENGTH,
)
model = JointConstraintModel(cfg).to(device)

if FREEZE_EPOCHS > 0:
    model.freeze_encoder()
    print(f"Encoder frozen for first {FREEZE_EPOCHS} epoch(s)")

total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Total parameters:     {total_params:,}")
print(f"Trainable parameters: {trainable_params:,}")

## 10. Training Loop
Joint training: NER + Obligation + CUAD Category with FP16 mixed precision.

In [ ]:
from torch.optim.lr_scheduler import CosineAnnealingLR
import time

optimiser = build_optimiser(model, ENCODER_LR, HEAD_LR, WEIGHT_DECAY)
scheduler = CosineAnnealingLR(optimiser, T_max=EPOCHS, eta_min=1e-6)

use_fp16 = USE_FP16 and device.type == "cuda"
scaler = torch.amp.GradScaler("cuda", enabled=use_fp16)

ckpt_dir = Path(OUTPUT_DIR) / "checkpoints" / "best_model"
best_f1 = 0.0
patience_cnt = 0
history = []

print(f"Training for up to {EPOCHS} epochs (patience={PATIENCE}, fp16={use_fp16})")
print("=" * 70)

for epoch in range(1, EPOCHS + 1):
    epoch_start = time.time()

    if epoch == FREEZE_EPOCHS + 1:
        model.unfreeze_encoder()
        print(f"\n>>> Encoder unfrozen at epoch {epoch}")

    # ── Train ─────────────────────────────────────────────────────────────
    model.train()
    total_loss = 0.0
    optimiser.zero_grad()

    for step, batch in enumerate(train_loader):
        batch = {k: v.to(device) for k, v in batch.items()}
        with torch.amp.autocast("cuda", enabled=use_fp16):
            result = model(
                input_ids=batch["input_ids"],
                attention_mask=batch["attention_mask"],
                ner_labels=batch["ner_labels"],
                obligation_labels=batch["obligation_labels"],
                category_labels=batch["category_labels"],
            )
            loss = result["loss"] / GRAD_ACCUM
        scaler.scale(loss).backward()
        total_loss += loss.item() * GRAD_ACCUM

        if (step + 1) % GRAD_ACCUM == 0:
            scaler.unscale_(optimiser)
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            scaler.step(optimiser)
            scaler.update()
            optimiser.zero_grad()

        if (step + 1) % 50 == 0:
            print(f"  Epoch {epoch} | Step {step+1}/{len(train_loader)} | Loss: {loss.item()*GRAD_ACCUM:.4f}")

    # Flush remaining gradients
    if len(train_loader) % GRAD_ACCUM != 0:
        scaler.unscale_(optimiser)
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        scaler.step(optimiser)
        scaler.update()
        optimiser.zero_grad()

    avg_loss = total_loss / len(train_loader)

    # ── Validate ──────────────────────────────────────────────────────────
    model.eval()
    all_ner_preds, all_ner_labels = [], []
    all_obl_preds, all_obl_labels = [], []
    all_cat_preds, all_cat_labels = [], []

    with torch.no_grad():
        for batch in val_loader:
            batch = {k: v.to(device) for k, v in batch.items()}
            ner_preds, obl_preds, cat_preds = model.predict(
                batch["input_ids"], batch["attention_mask"]
            )
            for i in range(ner_preds.shape[0]):
                all_ner_preds.append(ner_preds[i].cpu().tolist())
                all_ner_labels.append(batch["ner_labels"][i].cpu().tolist())
                all_obl_preds.append(obl_preds[i].cpu().tolist())
                all_obl_labels.append(batch["obligation_labels"][i].int().cpu().tolist())
                all_cat_preds.append(cat_preds[i].cpu().tolist())
                all_cat_labels.append(batch["category_labels"][i].int().cpu().tolist())

    ner_m = compute_ner_metrics(all_ner_preds, all_ner_labels)
    obl_m = compute_obligation_metrics(all_obl_preds, all_obl_labels)
    cat_m = compute_category_metrics(all_cat_preds, all_cat_labels)

    ner_f1 = ner_m.get("ner_f1", 0)
    obl_f1 = obl_m.get("obligation_micro_f1", 0)
    cat_f1 = cat_m.get("category_micro_f1", 0)
    combined_f1 = round((ner_f1 + obl_f1 + cat_f1) / 3, 4)

    scheduler.step()
    elapsed = time.time() - epoch_start

    entry = {
        "epoch": epoch, "train_loss": round(avg_loss, 4),
        "ner_f1": ner_f1, "obligation_f1": obl_f1,
        "category_f1": cat_f1, "combined_f1": combined_f1,
    }
    history.append(entry)

    print(f"\nEpoch {epoch:02d} ({elapsed:.0f}s) | Loss={avg_loss:.4f} | "
          f"NER_F1={ner_f1:.4f} | OBL_F1={obl_f1:.4f} | CAT_F1={cat_f1:.4f} | "
          f"Combined={combined_f1:.4f}")

    if combined_f1 > best_f1:
        best_f1 = combined_f1
        patience_cnt = 0
        save_checkpoint(model, tokenizer, cfg, ckpt_dir)
        print(f"  >>> New best model! Combined F1={best_f1:.4f}")
    else:
        patience_cnt += 1
        print(f"  No improvement ({patience_cnt}/{PATIENCE})")
        if patience_cnt >= PATIENCE:
            print("Early stopping triggered.")
            break

# Save training history
with open(Path(OUTPUT_DIR) / "training_history.json", "w") as f:
    json.dump(history, f, indent=2)

print("\n" + "=" * 70)
print(f"Training complete! Best combined F1: {best_f1:.4f}")
print(f"Checkpoint saved to: {ckpt_dir}")

## 11. Detailed Evaluation
Run full evaluation on validation set and print per-type metrics.

In [ ]:
# Load best checkpoint
state = torch.load(ckpt_dir / "constraint_model.pt", map_location=device, weights_only=True)
model.load_state_dict(state)
model.eval()
print("Loaded best checkpoint for evaluation")

# Re-run validation
all_ner_preds, all_ner_labels = [], []
all_obl_preds, all_obl_labels = [], []
all_cat_preds, all_cat_labels = [], []

with torch.no_grad():
    for batch in val_loader:
        batch = {k: v.to(device) for k, v in batch.items()}
        ner_preds, obl_preds, cat_preds = model.predict(
            batch["input_ids"], batch["attention_mask"]
        )
        for i in range(ner_preds.shape[0]):
            all_ner_preds.append(ner_preds[i].cpu().tolist())
            all_ner_labels.append(batch["ner_labels"][i].cpu().tolist())
            all_obl_preds.append(obl_preds[i].cpu().tolist())
            all_obl_labels.append(batch["obligation_labels"][i].int().cpu().tolist())
            all_cat_preds.append(cat_preds[i].cpu().tolist())
            all_cat_labels.append(batch["category_labels"][i].int().cpu().tolist())

all_metrics = {}
all_metrics.update(compute_ner_metrics(all_ner_preds, all_ner_labels))
all_metrics.update(compute_obligation_metrics(all_obl_preds, all_obl_labels))
all_metrics.update(compute_category_metrics(all_cat_preds, all_cat_labels))

print_metrics(all_metrics)

# Save full metrics
with open(Path(OUTPUT_DIR) / "eval_metrics.json", "w") as f:
    json.dump(all_metrics, f, indent=2)
print(f"\nFull metrics saved to {OUTPUT_DIR}/eval_metrics.json")

## 12. Demo: Constraint Extraction on Sample Text
Run the trained model on example legal text to see structured constraint output.

In [ ]:
# Create the extractor from the trained model
extractor = ConstraintExtractor(
    mode="hybrid",
    joint_model=model,
    tokenizer=tokenizer,
    device=device,
    ner_max_length=MAX_SEQ_LENGTH,
)

# Sample legal text for demo
demo_text = """
This Agreement is entered into as of January 15, 2024, by and between
Acme Corporation, a Delaware corporation ("the Company"), and John Smith
("the Consultant").

The Consultant shall provide consulting services to the Company for a
period of twelve (12) months commencing on the Effective Date. The Company
shall pay the Consultant a monthly fee of $10,000, payable within thirty
(30) days of receipt of each invoice.

The Consultant shall not, during the term of this Agreement and for a
period of two (2) years thereafter, directly or indirectly compete with
the Company in any business activity that is substantially similar to the
business of the Company.

This Agreement shall be governed by and construed in accordance with the
laws of the State of Delaware, without regard to its conflict of laws
principles. Any disputes arising under this Agreement shall be subject to
the exclusive jurisdiction of the courts of Delaware.

"Confidential Information" means any information disclosed by either party
to the other party that is designated as confidential or that reasonably
should be understood to be confidential given the nature of the information.
"""

# Create clause spans (simulating Stage 1 output)
import re as _re
paragraphs = [p.strip() for p in demo_text.strip().split("\n\n") if p.strip()]
clause_spans = []
offset = 0
for i, para in enumerate(paragraphs):
    start = demo_text.find(para, offset)
    end = start + len(para)
    clause_spans.append({
        "index": i,
        "text": para,
        "start_char": start,
        "end_char": end,
    })
    offset = end

# Extract constraints
result = extractor.extract(
    doc_id="demo_contract",
    clause_spans=clause_spans,
    full_text=demo_text,
)

# Display results
print(f"Document: {result.doc_id}")
print(f"Clauses: {len(result.clauses)}")
print(f"Parties: {result.all_parties}")
print(f"Dates: {result.all_dates}")
print(f"Jurisdictions: {result.all_jurisdictions}")
print()

for cc in result.clauses:
    print(f"--- Clause {cc.clause_index} ---")
    print(f"  Text (first 120 chars): {cc.clause_text[:120]}...")
    print(f"  Entities: {[e.entity_type.value + ': ' + e.text for e in cc.entities[:8]]}")
    print(f"  Obligations: {[o.obligation_type.value for o in cc.obligations]}")
    print(f"  CUAD categories: {cc.cuad_categories}")
    print(f"  Immutable: {cc.immutable}")
    print()

# Save full output
demo_output = result.to_dict()
with open(Path(OUTPUT_DIR) / "demo_output.json", "w", encoding="utf-8") as f:
    json.dump(demo_output, f, indent=2, ensure_ascii=False)
print(f"Demo output saved to {OUTPUT_DIR}/demo_output.json")

## 13. Extract Constraints for All Splits
Run the trained Stage 2 model on **all** Stage 1 predicted segments (train, val, test) so that Stage 3 can use constraint-aware prompting and evaluation across all splits.

In [ ]:
from tqdm.auto import tqdm

# extractor is already created from the demo cell above

split_paths = {
    "train": TRAIN_JSONL,
    "val":   VAL_JSONL,
    "test":  TEST_JSONL,
}

for split_name, jsonl_path in split_paths.items():
    if not Path(jsonl_path).exists():
        print(f"  SKIP {split_name}: {jsonl_path} not found")
        continue

    # Load Stage 1 predicted segments
    docs = []
    with open(jsonl_path, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if line:
                docs.append(json.loads(line))
    print(f"\n[{split_name}] {len(docs)} documents loaded")

    all_doc_constraints = []
    for doc in tqdm(docs, desc=f"Extracting constraints ({split_name})"):
        doc_id = doc["doc_id"]
        full_text = doc["text"]
        clauses_raw = doc.get("clauses", [])

        # Build clause spans from Stage 1 output
        clause_spans = []
        for ci, clause_info in enumerate(clauses_raw):
            if isinstance(clause_info, list):
                cs, ce = clause_info[0], clause_info[1]
                clause_text = full_text[cs:ce].strip()
            else:
                continue
            if clause_text:
                clause_spans.append({
                    "index": ci,
                    "text": clause_text,
                    "start_char": cs,
                    "end_char": ce,
                })

        if not clause_spans:
            continue

        result = extractor.extract(
            doc_id=doc_id,
            clause_spans=clause_spans,
            full_text=full_text,
        )
        all_doc_constraints.append(result.to_dict())

    # Save constraints for this split
    out_file = Path(OUTPUT_DIR) / f"{split_name}_constraints.json"
    with open(out_file, "w", encoding="utf-8") as f:
        json.dump({"documents": all_doc_constraints}, f, indent=2, ensure_ascii=False)

    total_clauses = sum(d["num_clauses"] for d in all_doc_constraints)
    print(f"  Saved {out_file.name}: {len(all_doc_constraints)} docs, {total_clauses} clauses")

print("\n✓ Constraints extracted for all splits.")

## 14. Package Output for Stage 3
Save the trained model, all split constraints, and config for use in Stage 3 (Simplification).

In [ ]:
import shutil

# Copy checkpoint to a clean output location
final_dir = Path(OUTPUT_DIR) / "final_model"
if ckpt_dir.exists():
    if final_dir.exists():
        shutil.rmtree(final_dir)
    shutil.copytree(ckpt_dir, final_dir)
    print(f"Final model copied to: {final_dir}")
    print(f"Contents: {list(final_dir.iterdir())}")
else:
    print("No checkpoint found — training may not have completed.")

print("\nStage 2 complete! Files saved:")
for f in sorted(Path(OUTPUT_DIR).rglob("*")):
    if f.is_file():
        size = f.stat().st_size
        print(f"  {f.relative_to(OUTPUT_DIR)}  ({size:,} bytes)")

print("\nKey outputs for Stage 3:")
print(f"  - train_constraints.json  (constraint profiles for train split)")
print(f"  - val_constraints.json    (constraint profiles for val split)")
print(f"  - test_constraints.json   (constraint profiles for test split)")
print(f"  - final_model/            (trained Stage 2 model)")
print(f"\nUpload the entire {OUTPUT_DIR} folder as a Kaggle dataset named 'stage2-output'.")